# V0 mass analysis — all plots preserved, stable sideband yields

This version keeps the **full plotting/output section from the original notebook**, including the multi-cut comparisons, CMS/STAR reference comparisons, efficiency/BR-corrected ratios, corrected spectra, and Tsallis/mT-scaled comparisons.

Only the yield-extraction core is changed:

- Yield extraction is **not taken from a Gaussian integral**.
- The native (unrebinned) mass histogram is used for the numerical yield.
- A left and a right sideband define either a **constant (`pol0`)** or **linear (`pol1`)** background density. The background under the signal window is obtained by interpolation/extrapolation between those sideband rectangles.
- `SIDEBAND_ORDER = 1` is the default. Both pol0 and pol1 yields are stored for QA.
- `YIELD_SOURCE = "like_subtracted"` uses unlike minus normalized like-sign first, then removes any residual sideband background. Change to `"foreground"` to test direct unlike-sign sideband subtraction.
- Mass rebinning is **QA/display only** and cannot change the extracted yield.
- pT edge bins are fractionally weighted, so small changes of pT boundaries do not abruptly add/remove a complete source bin.
- A dedicated sideband-QA PDF is produced for one cut. It shows the actual histogram, sideband regions, signal region, interpolated background, and a Gaussian **diagnostic only** (the Gaussian never defines the yield).

Because the input is a TH2 histogram, this cannot be literally unbinned. It is instead **rebin-independent counting at the native histogram binning**, with fractional treatment of window edges.

In [1]:

%run input/Format.ipynb
import ROOT as root
import math
root.gErrorIgnoreLevel = root.kFatal
%jsroot on
root.gStyle.SetOptStat(0)
root.gStyle.SetOptFit(0)


/home/yoren/.local/lib/python3.10/site-packages/nbformat/__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


Welcome to JupyROOT 6.30/06


Error in <TUnixSystem::FindDynamicLibrary>: input/logo/PHENIXTools/lib/libLogoPainter.so does not exist in /home/yoren/bnl/ROOT/install/lib:.:/home/yoren/bnl/ROOT/install/lib:/lib/x86_64-linux-gnu/glibc-hwcaps/x86-64-v3:/lib/x86_64-linux-gnu/glibc-hwcaps/x86-64-v2:/lib/x86_64-linux-gnu/tls/haswell/x86_64:/lib/x86_64-linux-gnu/tls/haswell:/lib/x86_64-linux-gnu/tls/x86_64:/lib/x86_64-linux-gnu/tls:/lib/x86_64-linux-gnu/haswell/x86_64:/lib/x86_64-linux-gnu/haswell:/lib/x86_64-linux-gnu/x86_64:/lib/x86_64-linux-gnu:/usr/lib/x86_64-linux-gnu/glibc-hwcaps/x86-64-v3:/usr/lib/x86_64-linux-gnu/glibc-hwcaps/x86-64-v2:/usr/lib/x86_64-linux-gnu/tls/haswell/x86_64:/usr/lib/x86_64-linux-gnu/tls/haswell:/usr/lib/x86_64-linux-gnu/tls/x86_64:/usr/lib/x86_64-linux-gnu/tls:/usr/lib/x86_64-linux-gnu/haswell/x86_64:/usr/lib/x86_64-linux-gnu/haswell:/usr/lib/x86_64-linux-gnu/x86_64:/usr/lib/x86_64-linux-gnu:/lib/glibc-hwcaps/x86-64-v3:/lib/glibc-hwcaps/x86-64-v2:/lib/tls/haswell/x86_64:/lib/tls/haswell:/lib

In [2]:
# ------------------------------------------------------------
# Efficiency pT bins
# ------------------------------------------------------------

N_EVENTS = 3800 * 10000 * 10

eff_pt_bins = [
    (0.5, 0.8),
    (0.8, 1.1),
    (1.1, 1.4),
    (1.4, 1.8),
    (1.8, 2.2),
    (2.2, 3.0),
    (3.0, 4.0)
]


# ============================================================
# Efficiencies IN PERCENT
# ============================================================

k0s_eff_percent = {
    "cut02_preselection": [(3.427,.110),(6.526,.261),(9.705,.331),(11.493,.361),(16.330,.445),(18.331,.492),(17.282,.534)],
    "cut03_baseline": [(3.255,.107),(6.196,.254),(9.269,.322),(10.966,.350),(15.873,.437),(17.692,.482),(16.863,.526)],
    "cut04_pairDCA_15mm": [(2.952,.101),(5.732,.243),(8.106,.301),(9.588,.327),(13.602,.406),(15.426,.450),(14.476,.485)],
    "cut05_pairDCA_10mm": [(2.537,.094),(4.933,.225),(7.043,.280),(8.270,.301),(11.421,.370),(13.191,.413),(12.574,.444)],
    "cut06_pairDCA_7mm": [(1.881,.081),(3.862,.199),(5.669,.248),(7.136,.277),(10.364,.347),(11.908,.387),(11.458,.422)],
    "cut07_pairDCA_5mm": [(1.380,.069),(2.931,.173),(4.568,.222),(5.992,.253),(9.015,.320),(10.430,.357),(9.773,.387)],
}

lambda_eff_percent = {
    "cut02_preselection": [None,(.842,.087),(2.820,.168),(6.263,.258),(11.770,.379),(12.766,.417),(11.675,.496)],
    "cut03_baseline": [None,(.792,.085),(2.721,.165),(6.067,.254),(11.515,.374),(12.381,.411),(11.674,.488)],
    "cut04_pairDCA_15mm": [None,(.667,.078),(2.391,.155),(5.358,.238),(9.956,.347),(10.455,.378),(9.930,.455)],
    "cut05_pairDCA_10mm": [None,(.547,.070),(1.994,.142),(4.481,.217),(8.517,.319),(8.997,.353),(8.418,.418)],
    "cut06_pairDCA_7mm": [None,(.426,.062),(1.556,.124),(3.797,.199),(7.518,.296),(7.796,.325),(7.838,.385)],
    "cut07_pairDCA_5mm": [None,(.330,.054),(1.317,.114),(3.231,.183),(6.377,.271),(6.567,.289),(7.099,.419)],
}

In [3]:

#input/output and labels

safe_to_pdf = False

file_path = "input/"
file_name = "v0s_pp_79513_prod_new_full_v0.root" # test_pp_0p75_1p7_skip_Chenxi_kf_v0 k0s_pp_1p23_1p0_1p6_Chenxi_kf_v0 New_pp_1.23_1p0_1vip6_my_380_6_2M.root 
#k0s_pp_1p00_0p85_1p7_Chenxi_kf_v0 k0s_pp_1p23_1p0_1p6_Chenxi_kf_v0_north k0s_pp_1p23_1p0_1p6_Chenxi_Masha_v0 k0s_pp_1p23_1p0_1p6_Chenxi_kf_v0_pphi
#k0s_pp_1p23_1p0_1p6_Chenxi_kf_v0_pdphi0p6 New_pp_1.00_0p85_1p7_my_380_6_2M_pdphi.root k0s_pp_1p23_1p0_1p6_Chenxi_kf_v0_pdphipt
#test_k0s_AuAu_0_0_Chenxi_kf_pdphipt test_OO_5p4_5p8_skip_Chenxi_kf_pdphipt.root k0s_pp_1p23_1p0_1p6_Chenxi_kf_v0_dphipt_4cm.
#k0s_pp_1p23_1p0_1p6_Chenxi_kf_v0_dphipt_3cm_0p2pt k0s_pp_1p23_1p0_1p6_Chenxi_Masha_v0_pdphipt_3cm_0p2pt
#k0s_pp_1p23_1p0_1p6_Chenxi_kf_v0_dphipt_3cm_0p2pt_noaplpha_pcaz15 k0s_pp_1p23_1p0_1p6_Chenxi_kf_v0_dphipt_3cm_0p2pt_noaplpha_pcaz15_q15_ntpc25 k0s_pp_1p23_1p0_1p6_Chenxi_kf_v0_dphipt_3cm_0p2pt_noaplpha_pcaz15_q15_ntpc30
#k0s_pp_1p23_1p0_1p6_Chenxi_kf_v0_dphipt_3cm_0p2pt_aplpha0p6_pcaz15_nptpc30_q15 k0s_pp_1p23_1p0_1p6_Chenxi_kf_v0_dphipt_3cm_0p2pt_aplpha0p6_pcaz15_nptpc30_q10
#k0s_pp_1p23_1p0_1p6_Chenxi_kf_v0_dphipt_3cm_0p2pt_noalpha k0s_pp_1p23_1p0_1p6_Chenxi_kf_v0_dphipt_2cm_0p2pt_noalpha
#new Masha
#k0s_pp_new_1p23_1p0_1p6_1p4_4_shift_v0_2M k0s_pp_new_1p23_1p0_1p6_000_v0_2M k0s_pp_new_1p23_1p0_1p6_000_v0_2M_X_v0
#k0s_pp_new_vtx_1p23_1p0_1p6_000_v0_2M k0s_pp_new_vtx_1p23_1p0_1p6_000_v0_2M_v0 k0s_pp_ratio_helix_new_vtx_1p23_1p0_1p6_000_v1_2M
#k0s_OO_ratio_kf_new_vtx_1p23_4p5_5p0_000_v0 k0s_AuAu_75391_ratio_kf_new_vtx_1p23_5p6_5p8_6cm_000_v0 k0s_AuAu_75405_ratio_kf_new_vtx_1p23_4p8_4p8_4cm_000_v0 k0s_OO_ratio_kf_new_vtx_1p23_4p5_5p0_3cm_000_v0
#k0s_pp_ratio_kf_new_vtx_1p23_1p0_1p6_2cm_000_v0_5M k0s_pp_ratio_kf_new_vtx_1p23_1p0_1p6_2cm_000_v0_7M 
#k0s_pp_nosailors_new_vtx_1p23_1p0_1p6_000_v0_5M k0s_pp_nosailors_nohighpt_new_vtx_1p23_1p0_1p6_000_v0_5M
#k0s_pp_ratio_kf_new_vtx_1p23_1p0_1p6_2cm_000_v0_7M v0s_pp_MYu_ratio_new_vtx_1p23_1p0_1p6_000_stream_full_corrmap_v6 v0s_pp_MYu_ratio_new_vtx_1p23_1p0_1p6_000_stream_full_mfield_v0 v0s_pp_MYu_ratio_new_vtx_1p23_1p0_1p6_000_stream_full_v1
#v0s_pp_79513_old_mfield_v1 v0s_pp_79513_old_mfield_noCS0_v0 v0s_pp_79513_old_mfield_q15_nc30_noCS0_v0 v0s_pp_79513_old_mfield_z15_q15_nc30_noCS0_v0 v0s_pp_79513_old_mfield_r4_z15_q15_nc30_noCS0_v0
#v0s_pp_79513_PR_full_pt02_r2_z15_q15_nc30_noCS0_v0 v0s_pp_79513_newv0_PR_full_pt02_r2_z15_q15_nc30_noCS0_v1 v0s_pp_79513_prod_new_full_v0 v0s_pp_79516_prod_new_full_v0 v0s_pp_79507_prod_new_full_v0
outdir = "output/v0_ratio"
root.gSystem.mkdir(outdir, True)
if "AuAu" in file_name:
    data_set = "Au+Au 2025"
    run_text = "Run 76905"
elif "75391" in file_name:
    data_set = "Au+Au 2025"
    run_text = "Run 75405"
elif "OO" in file_name:
    data_set = "O+O 2026"
    run_text = "Run 82626"
elif "75405" in file_name:
    data_set = "Au+Au 2025"
    run_text = "Run 75405"
else:
    data_set = "p+p 2025"
    run_text = "Run 79513"

like_background_method = "combined_like"

#available:
#"combined_like"
#"geometric"


In [4]:
# name, max_pca_z, max_dz, min_pt, min_lproj, max_alpha,
# max_pair_dca, min_dira, max_decay_r, min_ntpc_clusters

selections = [
    ("cut00_very_loose",       20.0, 1.50, 0.20, 2.0, 0.99, 6.00, 0.70, 20.0, 20),
    ("cut01_loose",            18.0, 1.00, 0.20, 2.0, 0.99, 4.00, 0.75, 20.0, 20),
    ("cut02_preselection",      15.0, 0.70, 0.20, 2.0, 0.99, 3.00, 0.80, 18.0, 25),
    ("cut03_baseline",          15.0, 0.50, 0.20, 2.0, 0.99, 2.00, 0.85, 15.0, 30),
    ("cut04_pairDCA_15mm",      12.0, 0.50, 0.20, 2.0, 0.99, 1.50, 0.88, 15.0, 30),
    ("cut05_pairDCA_10mm",      10.0, 0.40, 0.20, 2.0, 0.99, 1.00, 0.90, 13.0, 30),
    ("cut06_pairDCA_7mm",       10.0, 0.30, 0.20, 2.0, 0.99, 0.70, 0.93, 12.0, 32),
    ("cut07_pairDCA_5mm",       10.0, 0.20, 0.20, 2.0, 0.99, 0.50, 0.95, 10.0, 32),
    ("cut08_pairDCA_3mm",        8.0, 0.15, 0.20, 2.0, 0.99, 0.30, 0.98,  8.0, 35),
    ("cut09_pairDCA_2mm",        8.0, 0.10, 0.20, 2.0, 0.99, 0.20, 0.99,  6.0, 35),
]

cut_lines = {}
for name, max_z, max_dz, min_pt, min_lproj, _, max_pair_dca, min_dira, max_r, _ in selections:
    if run_text == "Run 75405":
        Lproj = 4.0
    cut_line = (
        f"|z_{{vtx}}|<{max_z:.0f} cm, "
        f"|PCA_{{z}}|<{max_dz:.1f} cm, |PCA_{{xy}}|<{max_pair_dca:.1f} cm, "
        f"DIRA>{min_dira:.2f}"
    )

    cut_lines[name] = cut_line
print(cut_lines)

{'cut00_very_loose': '|z_{vtx}|<20 cm, |PCA_{z}|<1.5 cm, |PCA_{xy}|<6.0 cm, DIRA>0.70', 'cut01_loose': '|z_{vtx}|<18 cm, |PCA_{z}|<1.0 cm, |PCA_{xy}|<4.0 cm, DIRA>0.75', 'cut02_preselection': '|z_{vtx}|<15 cm, |PCA_{z}|<0.7 cm, |PCA_{xy}|<3.0 cm, DIRA>0.80', 'cut03_baseline': '|z_{vtx}|<15 cm, |PCA_{z}|<0.5 cm, |PCA_{xy}|<2.0 cm, DIRA>0.85', 'cut04_pairDCA_15mm': '|z_{vtx}|<12 cm, |PCA_{z}|<0.5 cm, |PCA_{xy}|<1.5 cm, DIRA>0.88', 'cut05_pairDCA_10mm': '|z_{vtx}|<10 cm, |PCA_{z}|<0.4 cm, |PCA_{xy}|<1.0 cm, DIRA>0.90', 'cut06_pairDCA_7mm': '|z_{vtx}|<10 cm, |PCA_{z}|<0.3 cm, |PCA_{xy}|<0.7 cm, DIRA>0.93', 'cut07_pairDCA_5mm': '|z_{vtx}|<10 cm, |PCA_{z}|<0.2 cm, |PCA_{xy}|<0.5 cm, DIRA>0.95', 'cut08_pairDCA_3mm': '|z_{vtx}|<8 cm, |PCA_{z}|<0.1 cm, |PCA_{xy}|<0.3 cm, DIRA>0.98', 'cut09_pairDCA_2mm': '|z_{vtx}|<8 cm, |PCA_{z}|<0.1 cm, |PCA_{xy}|<0.2 cm, DIRA>0.99'}


In [5]:
cuts = ["cut00_very_loose", "cut01_loose", "cut02_preselection", "cut03_baseline", \
        "cut04_pairDCA_15mm", "cut05_pairDCA_10mm", "cut06_pairDCA_7mm", "cut07_pairDCA_5mm", \
            "cut08_pairDCA_3mm", "cut09_pairDCA_2mm" ]


In [6]:

#all analysis choices are controlled here

#one active cut for each particle
cuts_k0s = [
    "cut07_pairDCA_5mm", #0.5
    "cut07_pairDCA_5mm", #0.8
    "cut07_pairDCA_5mm", #1.1
    "cut07_pairDCA_5mm", #1.4
    "cut07_pairDCA_5mm",#1.8
    "cut07_pairDCA_5mm"  #2.2
]
cuts_lambda = [
    "cut03_baseline", #0.5
    "cut03_baseline", #0.8
    "cut03_baseline", #1.1
    "cut03_baseline", #1.4
    "cut03_baseline",#1.8
    "cut03_baseline"  #2.2
]

#AP plot selection
cut_ap = "cut04_pairDCA_15mm"
charge_ap = "unlike"

#pT bins
pt_k0s = [0.8, 1.1, 1.4, 1.8, 2.2, 3.0]
pt_lambda = [0.8, 1.1, 1.4, 1.8, 2.2, 3.0]

#rebin separately for each pT interval
rebin_k0s = [2,  2, 3, 4, 5, 5]
rebin_lambda = [2,  2, 3, 4, 5, 5]

#Lambda choice:
#"Lambda", "AntiLambda", or "LambdaPlusAntiLambda"
lambda_species = "LambdaPlusAntiLambda"
lambda_over_k0s_species = "LambdaPlusAntiLambda"

part_name_loc = "#Lambda + #bar{#Lambda}" if lambda_species == "LambdaPlusAntiLambda" else "#Lambda" if lambda_species == "Lambda" else "#bar{#Lambda}"

#mass drawing ranges
mass_range_k0s = (0.4, 0.6)
mass_range_lambda = (1.075, 1.165)

#like-sign normalization sidebands
sidebands_k0s = [(0.4, 0.44), (0.56, 0.62)]
sidebands_lambda = [(1.075, 1.100), (1.135, 1.165)]

#initial Gaussian-only fit ranges
gaus_range_k0s = (0.475, 0.520)
gaus_range_lambda = (1.105, 1.127)

#final Gaussian + pol2 fit ranges
fit_range_k0s = (0.43, 0.57)
fit_range_lambda = (1.090, 1.145)

#parameter limits
mean_limits_k0s = (0.485, 0.510)
sigma_limits_k0s = (0.002, 0.030)

mean_limits_lambda = (1.108, 1.123)
sigma_limits_lambda = (0.001, 0.020)

#canvas layout
ncols = 3

# ------------------------------------------------------------
# Stable yield-extraction controls
# ------------------------------------------------------------
# Numerical yield source:
#   "like_subtracted" = unlike - normalized like-sign, then sideband residual subtraction
#   "foreground"      = unlike directly, then sideband subtraction
YIELD_SOURCE = "like_subtracted"

# 0 = one constant density from both sidebands
# 1 = line through the average densities in the left/right sideband rectangles
SIDEBAND_ORDER = 1

# Rebin is ONLY for QA drawing. It never enters the numerical yield.
QA_REBIN = {"k0s": 2, "Lambda": 2, "AntiLambda": 2}

# One cut for detailed mass/background QA.
SIDEBAND_QA_CUT = "cut03_baseline"


In [7]:
def Format_hist_loc(h, title, x_title, y_title, x_title_size=0.06, y_title_size=0.06, x_title_offset=1.0, \
                    y_title_offset=1.0, label_size_x=0.06, label_size_y=0.06, label_size_z=0.06,\
                         z_axis_offset=1.0, z_range = None, center_title=False, title_size=0.06):
    if h is None:
        print("Error: Histogram is None. Cannot format.")
        return
    if title is not None:
        h.SetTitle(title)
    if x_title is not None:
        h.GetXaxis().SetTitle(x_title)
    if y_title is not None:
        h.GetYaxis().SetTitle(y_title)
    h.GetXaxis().SetTitleSize(x_title_size)
    h.GetYaxis().SetTitleSize(y_title_size)
    h.GetXaxis().SetTitleOffset(x_title_offset)
    h.GetYaxis().SetTitleOffset(y_title_offset)
    h.GetXaxis().SetLabelSize(label_size_x)
    h.GetYaxis().SetLabelSize(label_size_y)
    h.GetZaxis().SetLabelSize(label_size_z)
    h.GetZaxis().SetTitleOffset(z_axis_offset)
    #h.SetTitleSize(title_size)
    if z_range is not None:
        h.GetZaxis().SetRangeUser(z_range[0], z_range[1])
    if center_title:
        h.GetXaxis().CenterTitle()
        h.GetYaxis().CenterTitle()

In [8]:

#load file and helper functions

input_file = root.TFile.Open(file_path+file_name)

if not input_file or input_file.IsZombie():
    raise RuntimeError("Cannot open "+file_path+file_name)

ROOT_KEEPALIVE = []

def keep(obj):
    ROOT_KEEPALIVE.append(obj)
    root.SetOwnership(obj, False)
    return obj



In [9]:
def make_geometric_like_background(h_pp, h_mm, name):

    if h_pp.GetNbinsX() != h_mm.GetNbinsX():
        raise RuntimeError("++ and -- histograms have different binning")

    h_bg = h_pp.Clone(name)
    h_bg.Reset("ICES")
    h_bg.SetDirectory(0)
    keep(h_bg)

    for ibin in range(1, h_bg.GetNbinsX()+1):

        npp = h_pp.GetBinContent(ibin)
        nmm = h_mm.GetBinContent(ibin)

        epp = h_pp.GetBinError(ibin)
        emm = h_mm.GetBinError(ibin)

        if npp > 0. and nmm > 0.:

            value = 2.*math.sqrt(npp*nmm)

            #error propagation for B = 2*sqrt(N++*N--)
            error2 = (
                (nmm/npp)*epp*epp +
                (npp/nmm)*emm*emm
            )

            error = math.sqrt(error2)

        else:
            value = 0.
            error = 0.

        h_bg.SetBinContent(ibin, value)
        h_bg.SetBinError(ibin, error)

    return h_bg

In [10]:


def get_hist(path):
    h = input_file.Get(path)
    if not h:
        raise RuntimeError("Missing histogram: "+path)
    h = h.Clone(path.replace("/", "_")+"_clone")
    h.SetDirectory(0)
    return keep(h)

def hist_path(cut_name, charge_name, hist_name):
    return f"{cut_name}/{charge_name}/{hist_name}"

def sideband_integral(h, sidebands):
    total = 0.
    error2 = 0.

    for xmin, xmax in sidebands:
        b1 = h.GetXaxis().FindBin(xmin + 1.e-8)
        b2 = h.GetXaxis().FindBin(xmax - 1.e-8)

        for ibin in range(b1, b2 + 1):
            total += h.GetBinContent(ibin)
            error2 += h.GetBinError(ibin)**2

    return total, math.sqrt(error2)

def project_mass(h2, ptmin, ptmax, name):
    b1 = h2.GetXaxis().FindBin(ptmin+1.e-8)
    b2 = h2.GetXaxis().FindBin(ptmax-1.e-8)
    h = h2.ProjectionY(name, b1, b2, "e")
    h.SetDirectory(0)
    return keep(h)

def scale_like_to_unlike(h_unlike, h_like, sidebands):
    fg, fg_err = sideband_integral(h_unlike, sidebands)
    bg, bg_err = sideband_integral(h_like, sidebands)

    scale = fg/bg if bg > 0. else 1.
    scale_err = 0.

    if fg > 0. and bg > 0.:
        scale_err = scale*math.sqrt((fg_err/fg)**2+(bg_err/bg)**2)

    h_scaled = h_like.Clone(h_like.GetName()+"_scaled")
    h_scaled.SetDirectory(0)
    h_scaled.Scale(scale)
    keep(h_scaled)

    return h_scaled, scale, scale_err

def make_subtracted(h_unlike, h_like_scaled, name):
    h_sub = h_unlike.Clone(name)
    h_sub.SetDirectory(0)
    h_sub.Add(h_like_scaled, -1.)
    return keep(h_sub)

def setup_mass_hist(h, xtitle, ytitle="pairs"):
    Format_hist_loc(
        h,
        title="",
        x_title=xtitle,
        y_title=ytitle,
        x_title_offset=.9,
        y_title_offset=.9,
        center_title=True,
        x_title_size=0.055,
        y_title_size=0.055,
        label_size_x=0.045,
        label_size_y=0.045
    )

def draw_header(ptmin, ptmax, cut_name, extra=None, extra1 =None, x=0.16, size=0.06):
    latex = keep(root.TLatex())
    latex.SetNDC()
    latex.SetTextFont(42)

    latex.SetTextSize(size)
    latex.DrawLatex(x, 0.925, data_set+", "+run_text)

    latex.SetTextSize(size)
    latex.DrawLatex(x, 0.850, f"{ptmin:.1f} < p_{{T}}^{{V0}} < {ptmax:.1f} GeV/c")
    #latex.DrawLatex(x, 0.825, cut_name)

    if extra:
        latex.DrawLatex(x, 0.775, extra)

    if extra1:
        latex.DrawLatex(x, 0.700, extra1)

    return latex

def draw_zero_line(xmin, xmax):
    line = keep(root.TLine(xmin, 0., xmax, 0.))
    line.SetLineColor(root.kBlack)
    line.SetLineStyle(2)
    line.SetLineWidth(1)
    line.Draw("same")
    return line

def get_lambda_h2(cut_name, charge_name):
    if lambda_species == "Lambda":
        return get_hist(hist_path(cut_name, charge_name, "h_mass_Lambda_vs_v0_pt"))

    if lambda_species == "AntiLambda":
        return get_hist(hist_path(cut_name, charge_name, "h_mass_AntiLambda_vs_v0_pt"))

    h = get_hist(hist_path(cut_name, charge_name, "h_mass_Lambda_vs_v0_pt"))
    hbar = get_hist(hist_path(cut_name, charge_name, "h_mass_AntiLambda_vs_v0_pt"))
    h.Add(hbar)
    h.SetName(h.GetName()+"_LambdaPlusAntiLambda")
    return h


In [11]:
# ============================================================
# Yield extraction configuration
# ============================================================

# Change these pT boundaries freely. Numerical projection uses fractional
# overlap of source pT bins at the edges.
pt_bins_yield = [0.8, 1.1, 1.4, 1.8, 2.2, 3.0, 4.0, 5.0]

yield_csv = outdir + "/V0_yields_and_ratios_all_cuts.csv"
yield_root_file = outdir + "/V0_yields_and_ratios_all_cuts.root"
yield_ratio_pdf = outdir + "/V0_yields_and_ratios_all_cuts.pdf"
sideband_qa_pdf = outdir + f"/V0_sideband_QA_{SIDEBAND_QA_CUT}.pdf"

# The three windows are physical mass intervals, so their meaning does not
# change when the display histogram is rebinned.
# Tune these after looking at the QA PDF if desired.
yield_fit_config = {
    "k0s": {
        "hist_name": "h_mass_Kshort_vs_v0_pt",
        "norm_sidebands": sidebands_k0s,
        "left":   (0.400, 0.440),
        "signal": (0.440, 0.560),
        "right":  (0.56, 0.600),
        "draw_range": (0.40, 0.60),
        "mean_limits": mean_limits_k0s,
        "sigma_limits": sigma_limits_k0s,
        "sigma0": 0.010,
        "x_title": "M_{#pi^{+}#pi^{-}} [GeV/c^{2}]",
    },
    "Lambda": {
        "hist_name": "h_mass_Lambda_vs_v0_pt",
        "norm_sidebands": sidebands_lambda,
        "left":   (1.085, 1.102),
        "signal": (1.103, 1.129),
        "right":  (1.129, 1.160),
        "draw_range": (1.075, 1.165),
        "mean_limits": mean_limits_lambda,
        "sigma_limits": sigma_limits_lambda,
        "sigma0": 0.005,
        "x_title": "M_{p#pi^{-}} [GeV/c^{2}]",
    },
    "AntiLambda": {
        "hist_name": "h_mass_AntiLambda_vs_v0_pt",
        "norm_sidebands": sidebands_lambda,
        "left":   (1.085, 1.102),
        "signal": (1.103, 1.129),
        "right":  (1.129, 1.160),
        "draw_range": (1.075, 1.165),
        "mean_limits": mean_limits_lambda,
        "sigma_limits": sigma_limits_lambda,
        "sigma0": 0.005,
        "x_title": "M_{#bar{p}#pi^{+}} [GeV/c^{2}]",
    },
}

# Dedicated CMS/STAR comparison plots later in the notebook use the published
# quantity (Lambda+antiLambda)/(2 K0S).  This switch controls the generic
# Lambda/K0S-type quantity stored as row["lambda_over_k0s"].
#   "Lambda"                -> Lambda/K0S
#   "AntiLambda"            -> antiLambda/K0S
#   "LambdaPlusAntiLambda"  -> (Lambda+antiLambda)/(2 K0S)

from array import array
import csv


In [12]:
# ============================================================
# Compact, rebin-independent yield helpers
# ============================================================

_H2_CACHE = {}

def overlap_1d(lo1, hi1, lo2, hi2):
    return max(0.0, min(hi1, hi2) - max(lo1, lo2))


def get_species_h2(cut_name, charge_name, species):
    key = (cut_name, charge_name, species)
    if key not in _H2_CACHE:
        cfg = yield_fit_config[species]
        _H2_CACHE[key] = get_hist(hist_path(cut_name, charge_name, cfg["hist_name"]))
    return _H2_CACHE[key]


def project_mass_fractional(h2, ptmin, ptmax, name):
    """Project mass with fractional weighting of pT edge bins."""
    h = h2.ProjectionY(name, 1, h2.GetNbinsX(), "e")
    h.Reset("ICES")
    h.SetDirectory(0)
    keep(h)

    xax = h2.GetXaxis()
    for ix in range(1, h2.GetNbinsX() + 1):
        xlo = xax.GetBinLowEdge(ix)
        xhi = xax.GetBinUpEdge(ix)
        frac = overlap_1d(xlo, xhi, ptmin, ptmax) / (xhi - xlo)
        if frac <= 0.0:
            continue

        for iy in range(1, h2.GetNbinsY() + 1):
            gb = h2.GetBin(ix, iy)
            content = h.GetBinContent(iy) + frac * h2.GetBinContent(gb)
            err2 = h.GetBinError(iy)**2 + (frac * h2.GetBinError(gb))**2
            h.SetBinContent(iy, content)
            h.SetBinError(iy, math.sqrt(err2))

    return h


def integral_fractional(h, xmin, xmax):
    """Integral over a physical x range with fractional edge bins."""
    total = 0.0
    err2 = 0.0
    ax = h.GetXaxis()

    for ibin in range(1, h.GetNbinsX() + 1):
        blo = ax.GetBinLowEdge(ibin)
        bhi = ax.GetBinUpEdge(ibin)
        frac = overlap_1d(blo, bhi, xmin, xmax) / (bhi - blo)
        if frac <= 0.0:
            continue
        total += frac * h.GetBinContent(ibin)
        err2 += (frac * h.GetBinError(ibin))**2

    return total, math.sqrt(err2)


def integral_windows(h, windows):
    total = 0.0
    err2 = 0.0
    for lo, hi in windows:
        val, err = integral_fractional(h, lo, hi)
        total += val
        err2 += err*err
    return total, math.sqrt(err2)


def prepare_one_spectrum(cut_name, species, ptmin, ptmax, index):
    cfg = yield_fit_config[species]

    h_fg = project_mass_fractional(
        get_species_h2(cut_name, "unlike", species),
        ptmin, ptmax, f"h_{species}_{cut_name}_fg_pt{index}"
    )

    if like_background_method == "combined_like":
        h_like_raw = project_mass_fractional(
            get_species_h2(cut_name, "like", species),
            ptmin, ptmax, f"h_{species}_{cut_name}_like_pt{index}"
        )
    elif like_background_method == "geometric":
        h_pp = project_mass_fractional(
            get_species_h2(cut_name, "plusplus", species),
            ptmin, ptmax, f"h_{species}_{cut_name}_pp_pt{index}"
        )
        h_mm = project_mass_fractional(
            get_species_h2(cut_name, "minusminus", species),
            ptmin, ptmax, f"h_{species}_{cut_name}_mm_pt{index}"
        )
        h_like_raw = make_geometric_like_background(
            h_pp, h_mm, f"h_{species}_{cut_name}_geometric_pt{index}"
        )
    else:
        raise ValueError("like_background_method must be 'combined_like' or 'geometric'")

    fg_sb, fg_sb_err = integral_windows(h_fg, cfg["norm_sidebands"])
    bg_sb, bg_sb_err = integral_windows(h_like_raw, cfg["norm_sidebands"])
    scale = fg_sb / bg_sb if bg_sb > 0.0 else 1.0
    scale_err = 0.0
    if fg_sb > 0.0 and bg_sb > 0.0:
        scale_err = scale * math.sqrt((fg_sb_err/fg_sb)**2 + (bg_sb_err/bg_sb)**2)

    h_bg = h_like_raw.Clone(f"h_{species}_{cut_name}_like_scaled_pt{index}")
    h_bg.SetDirectory(0)
    h_bg.Scale(scale)
    keep(h_bg)

    h_sub = h_fg.Clone(f"h_{species}_{cut_name}_sub_pt{index}")
    h_sub.SetDirectory(0)
    h_sub.Add(h_bg, -1.0)
    keep(h_sub)

    return {
        "fg": h_fg,
        "like_raw": h_like_raw,
        "bg": h_bg,
        "sub": h_sub,
        "scale": scale,
        "scale_err": scale_err,
    }


def sideband_model(h, cfg, order):
    """
    Estimate background under cfg['signal'] from two sideband rectangles.

    order=0: one constant density from both sidebands.
    order=1: line through the average left/right sideband densities.
    """
    if order not in (0, 1):
        raise ValueError("SIDEBAND_ORDER must be 0 or 1")

    llo, lhi = cfg["left"]
    slo, shi = cfg["signal"]
    rlo, rhi = cfg["right"]

    nl, el = integral_fractional(h, llo, lhi)
    nr, er = integral_fractional(h, rlo, rhi)

    wl, wr, ws = lhi-llo, rhi-rlo, shi-slo
    xl, xr, xs = 0.5*(llo+lhi), 0.5*(rlo+rhi), 0.5*(slo+shi)

    dl, dr = nl/wl, nr/wr
    edl, edr = el/wl, er/wr

    if order == 0:
        density = (nl + nr) / (wl + wr)
        density_err = math.sqrt(el*el + er*er) / (wl + wr)
        p0, p1 = density, 0.0
        bg = density * ws
        bg_err = density_err * ws
    else:
        t = (xs - xl) / (xr - xl)
        density = (1.0-t)*dl + t*dr
        density_err = math.sqrt(((1.0-t)*edl)**2 + (t*edr)**2)
        p1 = (dr - dl) / (xr - xl)
        p0 = dl - p1*xl
        bg = density * ws
        bg_err = density_err * ws

    return {
        "order": order,
        "left_count": nl, "left_error": el,
        "right_count": nr, "right_error": er,
        "left_density": dl, "right_density": dr,
        "p0": p0, "p1": p1,
        "background": bg, "background_error": bg_err,
    }


def density_clone(h, name):
    d = h.Clone(name)
    d.SetDirectory(0)
    keep(d)
    for ibin in range(1, d.GetNbinsX()+1):
        width = d.GetXaxis().GetBinWidth(ibin)
        if width > 0.0:
            d.SetBinContent(ibin, d.GetBinContent(ibin)/width)
            d.SetBinError(ibin, d.GetBinError(ibin)/width)
    return d


def diagnostic_gaussian(h_source, cfg, bg, name):
    """Gaussian QA only. It never enters the numerical yield."""
    h_sig = density_clone(h_source, name + "_signal_density")
    slo, shi = cfg["signal"]

    for ibin in range(1, h_sig.GetNbinsX()+1):
        x = h_sig.GetBinCenter(ibin)
        b = bg["p0"] + bg["p1"]*x
        h_sig.SetBinContent(ibin, h_sig.GetBinContent(ibin) - b)

    gaus = keep(root.TF1(name + "_gaus", "gaus", slo, shi))
    b1 = h_sig.GetXaxis().FindBin(slo + 1e-12)
    b2 = h_sig.GetXaxis().FindBin(shi - 1e-12)
    peak_bin = max(range(b1, b2+1), key=lambda b: h_sig.GetBinContent(b))
    amp0 = max(h_sig.GetBinContent(peak_bin), 1.0)
    mean0 = min(max(h_sig.GetBinCenter(peak_bin), cfg["mean_limits"][0]), cfg["mean_limits"][1])

    gaus.SetParameters(amp0, mean0, cfg["sigma0"])
    gaus.SetParLimits(1, cfg["mean_limits"][0], cfg["mean_limits"][1])
    gaus.SetParLimits(2, cfg["sigma_limits"][0], cfg["sigma_limits"][1])

    fit_ptr = h_sig.Fit(gaus, "RQS0")
    status = int(fit_ptr)
    ndf = gaus.GetNDF()

    full_formula = "gaus(0)+pol1(3)" if bg["order"] == 1 else "gaus(0)+pol0(3)"
    full = keep(root.TF1(name + "_qa_model", full_formula, *cfg["draw_range"]))
    full.SetParameter(0, gaus.GetParameter(0))
    full.SetParameter(1, gaus.GetParameter(1))
    full.SetParameter(2, abs(gaus.GetParameter(2)))
    full.SetParameter(3, bg["p0"])
    if bg["order"] == 1:
        full.SetParameter(4, bg["p1"])

    return {
        "gaus": gaus,
        "full": full,
        "mean": gaus.GetParameter(1),
        "mean_error": gaus.GetParError(1),
        "sigma": abs(gaus.GetParameter(2)),
        "sigma_error": gaus.GetParError(2),
        "fit_status": status,
        "chi2_ndf": gaus.GetChisquare()/ndf if ndf > 0 else float("nan"),
    }


def extract_sideband_yield(spectrum, species, name):
    cfg = yield_fit_config[species]
    h_source = spectrum["sub"] if YIELD_SOURCE == "like_subtracted" else spectrum["fg"]

    models = {order: sideband_model(h_source, cfg, order) for order in (0, 1)}
    chosen = models[SIDEBAND_ORDER]

    sig_sum, sig_err = integral_fractional(h_source, *cfg["signal"])
    value = sig_sum - chosen["background"]

    # The histogram already contains fg and scaled-like statistical errors.
    # Propagate the common like-sign normalization-scale uncertainty through
    # the *same full sideband functional* (signal window minus interpolated BG),
    # so the signal and sideband pieces keep their proper correlation.
    scale_norm_err = {0: 0.0, 1: 0.0}
    if YIELD_SOURCE == "like_subtracted":
        like_sig, _ = integral_fractional(spectrum["like_raw"], *cfg["signal"])
        for order in (0, 1):
            like_model = sideband_model(spectrum["like_raw"], cfg, order)
            dY_dscale = -(like_sig - like_model["background"])
            scale_norm_err[order] = abs(dY_dscale * spectrum["scale_err"])

    error = math.sqrt(
        sig_err**2 + chosen["background_error"]**2 + scale_norm_err[SIDEBAND_ORDER]**2
    )

    # Alternative pol0/pol1 values are retained as an immediate background-method QA.
    alt = {}
    for order, model in models.items():
        alt_value = sig_sum - model["background"]
        alt_error = math.sqrt(
            sig_err**2 + model["background_error"]**2 + scale_norm_err[order]**2
        )
        alt[order] = (alt_value, alt_error)

    diag = diagnostic_gaussian(h_source, cfg, chosen, name)

    bg_func = keep(root.TF1(
        name + "_sideband_bg",
        "pol1" if SIDEBAND_ORDER == 1 else "pol0",
        *cfg["draw_range"]
    ))
    bg_func.SetParameter(0, chosen["p0"])
    if SIDEBAND_ORDER == 1:
        bg_func.SetParameter(1, chosen["p1"])

    return {
        **spectrum,
        "source": h_source,
        "yield": value,
        "yield_error": error,
        "yield_pol0": alt[0][0], "yield_pol0_error": alt[0][1],
        "yield_pol1": alt[1][0], "yield_pol1_error": alt[1][1],
        "sideband_systematic": abs(alt[1][0] - alt[0][0]),
        "signal_window_sum": sig_sum,
        "signal_window_error": sig_err,
        "residual_bg_sum": chosen["background"],
        "residual_bg_error": chosen["background_error"],
        "background": bg_func,
        "full": diag["full"],
        "signal": diag["gaus"],
        "mean": diag["mean"],
        "mean_error": diag["mean_error"],
        "sigma": diag["sigma"],
        "sigma_error": diag["sigma_error"],
        "fit_status": diag["fit_status"],
        "chi2_ndf": diag["chi2_ndf"],
        "window": cfg["signal"],
        "left_window": cfg["left"],
        "right_window": cfg["right"],
        "models": models,
    }


def ratio_with_error(numerator, numerator_error, denominator, denominator_error):
    if not all(math.isfinite(v) for v in [numerator, numerator_error, denominator, denominator_error]):
        return float("nan"), float("nan")
    if denominator <= 0.0:
        return float("nan"), float("nan")
    ratio = numerator / denominator
    if numerator == 0.0:
        return ratio, abs(numerator_error / denominator)
    error = abs(ratio) * math.sqrt(
        (numerator_error/numerator)**2 + (denominator_error/denominator)**2
    )
    return ratio, error


def selected_baryon_yield(lambda_yield, lambda_error, antilambda_yield, antilambda_error):
    if lambda_over_k0s_species == "Lambda":
        return lambda_yield, lambda_error
    if lambda_over_k0s_species == "AntiLambda":
        return antilambda_yield, antilambda_error
    if lambda_over_k0s_species == "LambdaPlusAntiLambda":
        return (
            0.5*(lambda_yield + antilambda_yield),
            0.5*math.sqrt(lambda_error**2 + antilambda_error**2)
        )
    raise ValueError("lambda_over_k0s_species must be Lambda, AntiLambda, or LambdaPlusAntiLambda")


def make_graph(name, title, rows, value_key, error_key):
    good = [
        row for row in rows
        if math.isfinite(row.get(value_key, float("nan")))
        and math.isfinite(row.get(error_key, float("nan")))
    ]
    graph = keep(root.TGraphErrors(
        len(good),
        array("d", [row["pt_center"] for row in good]),
        array("d", [row[value_key] for row in good]),
        array("d", [row["pt_half_width"] for row in good]),
        array("d", [row[error_key] for row in good]),
    ))
    graph.SetName(name)
    graph.SetTitle(title)
    graph.SetMarkerStyle(20)
    graph.SetMarkerSize(1.0)
    graph.SetLineWidth(2)
    return graph


def effective_efficiency(eff_table, cut_name, ptmin, ptmax):
    """Width-weighted efficiency over a data pT interval; returns fractions."""
    values = eff_table.get(cut_name)
    if values is None or ptmax <= ptmin:
        return None

    width = ptmax - ptmin
    weighted = 0.0
    err2 = 0.0
    covered = 0.0

    for (elo, ehi), entry in zip(eff_pt_bins, values):
        w = overlap_1d(ptmin, ptmax, elo, ehi)
        if w <= 0.0:
            continue
        if entry is None:
            return None
        eff, err = entry
        weighted += w*eff
        err2 += (w*err)**2
        covered += w

    if abs(covered-width) > 1e-9*max(1.0, width):
        return None

    return (weighted/width)/100.0, (math.sqrt(err2)/width)/100.0


In [13]:
# ============================================================
# Extract all yields ONCE. Every later plot reads these rows.
# ============================================================

all_cut_results = {}
flat_rows = []
fit_objects_by_cut = {}

for cut_name in cuts:
    print("\nProcessing", cut_name)
    rows = []
    fit_objects_by_cut[cut_name] = {}

    for ipt, (ptmin, ptmax) in enumerate(zip(pt_bins_yield[:-1], pt_bins_yield[1:])):
        species_results = {}

        for species in ["k0s", "Lambda", "AntiLambda"]:
            spectrum = prepare_one_spectrum(cut_name, species, ptmin, ptmax, ipt)
            result = extract_sideband_yield(
                spectrum, species, f"qa_{species}_{cut_name}_pt{ipt}"
            )
            species_results[species] = result
            fit_objects_by_cut[cut_name][(species, ipt)] = result

        ky, ke = species_results["k0s"]["yield"], species_results["k0s"]["yield_error"]
        ly, le = species_results["Lambda"]["yield"], species_results["Lambda"]["yield_error"]
        ay, ae = species_results["AntiLambda"]["yield"], species_results["AntiLambda"]["yield_error"]

        anti_ratio, anti_ratio_error = ratio_with_error(ay, ae, ly, le)
        baryon_yield, baryon_error = selected_baryon_yield(ly, le, ay, ae)
        lambda_over_k0s, lambda_over_k0s_error = ratio_with_error(
            baryon_yield, baryon_error, ky, ke
        )

        average_yield = 0.5*(ly + ay)
        average_error = 0.5*math.sqrt(le*le + ae*ae)
        average_ratio, average_ratio_error = ratio_with_error(
            average_yield, average_error, ky, ke
        )

        row = {
            "cut": cut_name,
            "ptmin": ptmin,
            "ptmax": ptmax,
            "pt_center": 0.5*(ptmin+ptmax),
            "pt_half_width": 0.5*(ptmax-ptmin),
            "antilambda_over_lambda": anti_ratio,
            "antilambda_over_lambda_error": anti_ratio_error,
            "lambda_over_k0s": lambda_over_k0s,
            "lambda_over_k0s_error": lambda_over_k0s_error,
            "average_lambda_over_k0s": average_ratio,
            "average_lambda_over_k0s_error": average_ratio_error,
        }

        for species, prefix in [("k0s", "k0s"), ("Lambda", "lambda"), ("AntiLambda", "antilambda")]:
            r = species_results[species]
            row.update({
                f"{prefix}_yield": r["yield"],
                f"{prefix}_yield_error": r["yield_error"],
                f"{prefix}_yield_pol0": r["yield_pol0"],
                f"{prefix}_yield_pol1": r["yield_pol1"],
                f"{prefix}_sideband_systematic": r["sideband_systematic"],
                f"{prefix}_residual_bg": r["residual_bg_sum"],
                f"{prefix}_mean": r["mean"],
                f"{prefix}_sigma": r["sigma"],
                f"{prefix}_fit_status": r["fit_status"],
                f"{prefix}_chi2_ndf": r["chi2_ndf"],
            })

        rows.append(row)
        flat_rows.append(row)

        print(
            f"  {ptmin:.1f}-{ptmax:.1f}: "
            f"K0S={ky:.0f}+/-{ke:.0f}, Lambda={ly:.0f}+/-{le:.0f}, "
            f"antiLambda={ay:.0f}+/-{ae:.0f}, anti/Lambda={anti_ratio:.3f}, "
            f"selected/K0S={lambda_over_k0s:.3f}"
        )

    all_cut_results[cut_name] = rows

# Guard against accidental redefinition of antiLambda/Lambda later.
for cut_name, rows in all_cut_results.items():
    for row in rows:
        if row["lambda_yield"] > 0.0 and math.isfinite(row["antilambda_over_lambda"]):
            direct = row["antilambda_yield"] / row["lambda_yield"]
            assert abs(direct-row["antilambda_over_lambda"]) < 1e-12*max(1.0, abs(direct))

print("\nYield method:", YIELD_SOURCE, "+ sideband pol" + str(SIDEBAND_ORDER))
print("Mass rebinning does not enter the numerical yield.")



Processing cut00_very_loose
  0.8-1.1: K0S=359352+/-1415, Lambda=26767+/-287, antiLambda=15997+/-241, anti/Lambda=0.598, selected/K0S=0.060
  1.1-1.4: K0S=226882+/-1196, Lambda=40650+/-330, antiLambda=28194+/-305, anti/Lambda=0.694, selected/K0S=0.152
  1.4-1.8: K0S=188182+/-1143, Lambda=60457+/-384, antiLambda=48794+/-369, anti/Lambda=0.807, selected/K0S=0.290
  1.8-2.2: K0S=116624+/-914, Lambda=44552+/-346, antiLambda=38655+/-348, anti/Lambda=0.868, selected/K0S=0.357
  2.2-3.0: K0S=82214+/-747, Lambda=28396+/-285, antiLambda=25095+/-302, anti/Lambda=0.884, selected/K0S=0.325
  3.0-4.0: K0S=17733+/-368, Lambda=4807+/-132, antiLambda=4175+/-153, anti/Lambda=0.869, selected/K0S=0.253
  4.0-5.0: K0S=2207+/-166, Lambda=505+/-53, antiLambda=336+/-68, anti/Lambda=0.665, selected/K0S=0.190

Processing cut01_loose
  0.8-1.1: K0S=335080+/-1349, Lambda=24561+/-273, antiLambda=14719+/-230, anti/Lambda=0.599, selected/K0S=0.059
  1.1-1.4: K0S=211988+/-1143, Lambda=37037+/-315, antiLambda=25907+

In [14]:
# ============================================================
# Save numerical results to CSV and ROOT
# ============================================================

csv_columns = [
    "cut", "ptmin", "ptmax",
    "k0s_yield", "k0s_yield_error", "k0s_yield_pol0", "k0s_yield_pol1", "k0s_sideband_systematic",
    "k0s_mean", "k0s_sigma", "k0s_fit_status", "k0s_chi2_ndf",
    "lambda_yield", "lambda_yield_error", "lambda_yield_pol0", "lambda_yield_pol1", "lambda_sideband_systematic",
    "lambda_mean", "lambda_sigma", "lambda_fit_status", "lambda_chi2_ndf",
    "antilambda_yield", "antilambda_yield_error", "antilambda_yield_pol0", "antilambda_yield_pol1", "antilambda_sideband_systematic",
    "antilambda_mean", "antilambda_sigma", "antilambda_fit_status", "antilambda_chi2_ndf",
    "antilambda_over_lambda", "antilambda_over_lambda_error",
    "lambda_over_k0s", "lambda_over_k0s_error",
    "average_lambda_over_k0s", "average_lambda_over_k0s_error",
]

with open(yield_csv, "w", newline="") as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=csv_columns)
    writer.writeheader()
    for row in flat_rows:
        writer.writerow({column: row.get(column, float("nan")) for column in csv_columns})

output_root = root.TFile.Open(yield_root_file, "RECREATE")
graphs_by_cut = {}

for cut_name, rows in all_cut_results.items():
    output_root.mkdir(cut_name)
    output_root.cd(cut_name)

    graphs = {
        "k0s_yield": make_graph(f"g_k0s_yield_{cut_name}", ";p_{T} [GeV/c];K^{0}_{S} yield", rows, "k0s_yield", "k0s_yield_error"),
        "lambda_yield": make_graph(f"g_lambda_yield_{cut_name}", ";p_{T} [GeV/c];#Lambda yield", rows, "lambda_yield", "lambda_yield_error"),
        "antilambda_yield": make_graph(f"g_antilambda_yield_{cut_name}", ";p_{T} [GeV/c];#bar{#Lambda} yield", rows, "antilambda_yield", "antilambda_yield_error"),
        "antilambda_over_lambda": make_graph(f"g_antilambda_over_lambda_{cut_name}", ";p_{T} [GeV/c];#bar{#Lambda}/#Lambda", rows, "antilambda_over_lambda", "antilambda_over_lambda_error"),
        "lambda_over_k0s": make_graph(f"g_lambda_over_k0s_{cut_name}", ";p_{T} [GeV/c];selected baryon/K^{0}_{S}", rows, "lambda_over_k0s", "lambda_over_k0s_error"),
    }

    for graph in graphs.values():
        graph.Write()
    graphs_by_cut[cut_name] = graphs

output_root.Close()
print("Saved CSV :", yield_csv)
print("Saved ROOT:", yield_root_file)


Saved CSV : output/v0_ratio/V0_yields_and_ratios_all_cuts.csv
Saved ROOT: output/v0_ratio/V0_yields_and_ratios_all_cuts.root


In [15]:
# ============================================================
# Sideband QA: one multipage PDF for one cut
# ============================================================
# Each page is one species; each pad is one pT bin.
# Black points: the actual yield-source histogram.
# Blue dashed: sideband-interpolated background.
# Red: background + diagnostic Gaussian (visual QA only).
# Gray/orange boxes: left/right sidebands; green box: signal window.


def _qa_density_hist(h_counts, name, rebin):
    h = h_counts.Clone(name)
    h.SetDirectory(0)
    keep(h)
    if rebin > 1:
        h.Rebin(rebin)
    for ibin in range(1, h.GetNbinsX()+1):
        width = h.GetXaxis().GetBinWidth(ibin)
        if width > 0.0:
            h.SetBinContent(ibin, h.GetBinContent(ibin)/width)
            h.SetBinError(ibin, h.GetBinError(ibin)/width)
    return h


def draw_sideband_qa(cut_name):
    canvases = {}
    npt = len(pt_bins_yield)-1
    ncols = 3
    nrows = math.ceil(npt/ncols)

    for species in ["k0s", "Lambda", "AntiLambda"]:
        cfg = yield_fit_config[species]
        c = keep(root.TCanvas(
            f"c_sideband_qa_{species}_{cut_name}",
            f"{species} sideband QA", 1500, 420*nrows
        ))
        c.Divide(ncols, nrows)

        for ipt, (ptmin, ptmax) in enumerate(zip(pt_bins_yield[:-1], pt_bins_yield[1:])):
            c.cd(ipt+1)
            root.gPad.SetLeftMargin(0.14)
            root.gPad.SetBottomMargin(0.13)
            root.gPad.SetTopMargin(0.08)

            r = fit_objects_by_cut[cut_name][(species, ipt)]
            h = _qa_density_hist(
                r["source"], f"h_qa_draw_{species}_{cut_name}_{ipt}", QA_REBIN[species]
            )
            h.GetXaxis().SetRangeUser(*cfg["draw_range"])
            setup_mass_hist(h, cfg["x_title"], "pairs / (GeV/c^{2})")
            h.SetMarkerStyle(20)
            h.SetMarkerSize(0.55)

            ymin = min(0.0, h.GetMinimum())
            ymax = max(1.0, h.GetMaximum()) * 1.16
            h.SetMinimum(ymin - 0.06*(ymax-ymin))
            h.SetMaximum(ymax)
            h.Draw("E")

            # Window shading.
            for window, color in [
                (r["left_window"], root.kGray+1),
                (r["right_window"], root.kOrange-3),
                (r["window"], root.kGreen-9),
            ]:
                box = keep(root.TBox(window[0], h.GetMinimum(), window[1], h.GetMaximum()))
                box.SetFillColorAlpha(color, 0.16)
                box.SetLineColor(color)
                box.Draw("SAME")

            h.Draw("E SAME")

            r["background"].SetLineColor(root.kBlue+1)
            r["background"].SetLineStyle(2)
            r["background"].SetLineWidth(2)
            r["background"].Draw("SAME")

            r["full"].SetLineColor(root.kRed+1)
            r["full"].SetLineWidth(2)
            r["full"].Draw("SAME")

            text = keep(root.TLatex())
            text.SetNDC()
            text.SetTextFont(42)
            text.SetTextSize(0.034)
            text.DrawLatex(0.16, 0.92, f"{ptmin:.1f}<p_{{T}}<{ptmax:.1f} GeV/c")
            text.SetTextSize(0.028)
            text.DrawLatex(0.16, 0.86, f"Y(pol{SIDEBAND_ORDER})={r['yield']:.0f} #pm {r['yield_error']:.0f}")
            text.DrawLatex(0.16, 0.81, f"Y(pol0)={r['yield_pol0']:.0f}, Y(pol1)={r['yield_pol1']:.0f}")
            text.DrawLatex(0.16, 0.76, f"residual BG={r['residual_bg_sum']:.0f}, #Delta_{{01}}={r['sideband_systematic']:.0f}")
            text.DrawLatex(0.16, 0.71, f"diag #mu={r['mean']:.5f}, #sigma={1000*r['sigma']:.1f} MeV")
            text.DrawLatex(0.16, 0.66, f"diag fit status={r['fit_status']}, #chi^{{2}}/ndf={r['chi2_ndf']:.2f}")

        canvases[species] = c

    ordered = [canvases[s] for s in ["k0s", "Lambda", "AntiLambda"]]
    for i, c in enumerate(ordered):
        suffix = "(" if i == 0 else ")" if i == len(ordered)-1 else ""
        c.Print(sideband_qa_pdf + suffix)

    print("Saved sideband QA:", sideband_qa_pdf)
    return canvases

sideband_qa_canvases = draw_sideband_qa(SIDEBAND_QA_CUT)
sideband_qa_canvases["k0s"].Draw()


Saved sideband QA: output/v0_ratio/V0_sideband_QA_cut03_baseline.pdf


In [16]:
# ============================================================
# One PDF page per cut:
#   pad 1: K0S, Lambda, anti-Lambda yields
#   pad 2: anti-Lambda / Lambda
#   pad 3: Lambda / K0S
#   pad 4: cut definition and fit-status summary
# ============================================================

pdf_open = yield_ratio_pdf + "["
pdf_close = yield_ratio_pdf + "]"

pdf_canvas = root.TCanvas(
    "c_v0_yields_ratios_by_cut",
    "c_v0_yields_ratios_by_cut",
    1500,
    1100
)
pdf_canvas.Print(pdf_open)

plot_keepalive = []

for cut_index, cut_name in enumerate(cuts):
    pdf_canvas.Clear()
    pdf_canvas.Divide(2, 2)

    rows = all_cut_results[cut_name]
    graphs = graphs_by_cut[cut_name]

    # --------------------------------------------------------
    # Pad 1: all three yields
    # --------------------------------------------------------
    pdf_canvas.cd(1)
    root.gPad.SetLeftMargin(0.14)
    root.gPad.SetBottomMargin(0.13)
    root.gPad.SetRightMargin(0.04)
    root.gPad.SetTopMargin(0.10)
    root.gPad.SetLogy()

    g_k0s = graphs["k0s_yield"]
    g_lambda = graphs["lambda_yield"]
    g_antilambda = graphs["antilambda_yield"]

    g_k0s.SetMarkerColor(root.kBlack)
    g_k0s.SetLineColor(root.kBlack)

    g_lambda.SetMarkerColor(root.kBlue + 1)
    g_lambda.SetLineColor(root.kBlue + 1)
    g_lambda.SetMarkerStyle(21)

    g_antilambda.SetMarkerColor(root.kRed + 1)
    g_antilambda.SetLineColor(root.kRed + 1)
    g_antilambda.SetMarkerStyle(22)

    ymax = max(
        max(row["k0s_yield"] for row in rows),
        max(row["lambda_yield"] for row in rows),
        max(row["antilambda_yield"] for row in rows)
    )

    frame_yield = keep(root.TH1F(
        f"frame_yield_{cut_index}",
        "",
        100,
        pt_bins_yield[0],
        pt_bins_yield[-1]
    ))
    frame_yield.SetDirectory(0)
    frame_yield.SetMinimum(max(0.5, ymax * 1.0e-4))
    frame_yield.SetMaximum(max(1.0, 3.0 * ymax))
    Format_hist_loc(
        frame_yield,
        "",
        "p_{T} [GeV/c]",
        "raw sideband-subtracted yield",
        center_title=True,
        x_title_offset=1.0,
        y_title_offset=1.05
    )
    frame_yield.Draw()

    g_k0s.Draw("P SAME")
    g_lambda.Draw("P SAME")
    g_antilambda.Draw("P SAME")

    legend_yield = keep(root.TLegend(0.58, 0.67, 0.90, 0.88))
    legend_yield.SetBorderSize(0)
    legend_yield.SetFillStyle(0)
    legend_yield.AddEntry(g_k0s, "K^{0}_{S}", "lp")
    legend_yield.AddEntry(g_lambda, "#Lambda", "lp")
    legend_yield.AddEntry(g_antilambda, "#bar{#Lambda}", "lp")
    legend_yield.Draw()

    latex = keep(root.TLatex())
    latex.SetNDC()
    latex.SetTextFont(42)
    latex.SetTextSize(0.045)
    latex.DrawLatex(0.16, 0.92, data_set + ", " + run_text)
    latex.DrawLatex(0.16, 0.86, cut_name)

    # --------------------------------------------------------
    # Pad 2: anti-Lambda / Lambda
    # --------------------------------------------------------
    pdf_canvas.cd(2)
    root.gPad.SetLeftMargin(0.14)
    root.gPad.SetBottomMargin(0.13)
    root.gPad.SetRightMargin(0.04)
    root.gPad.SetTopMargin(0.10)

    g_ratio_anti = graphs["antilambda_over_lambda"]
    g_ratio_anti.SetMarkerColor(root.kBlack)
    g_ratio_anti.SetLineColor(root.kBlack)

    valid_anti = [
        row["antilambda_over_lambda"]
        for row in rows
        if math.isfinite(row["antilambda_over_lambda"])
    ]
    anti_max = max(valid_anti) if valid_anti else 1.0

    frame_anti = keep(root.TH1F(
        f"frame_anti_ratio_{cut_index}",
        "",
        100,
        pt_bins_yield[0],
        pt_bins_yield[-1]
    ))
    frame_anti.SetDirectory(0)
    frame_anti.SetMinimum(0.0)
    frame_anti.SetMaximum(max(1.5, 1.4 * anti_max))
    Format_hist_loc(
        frame_anti,
        "",
        "p_{T} [GeV/c]",
        "#bar{#Lambda}/#Lambda",
        center_title=True,
        x_title_offset=1.0,
        y_title_offset=1.05
    )
    frame_anti.Draw()
    g_ratio_anti.Draw("P SAME")

    line_one_anti = keep(root.TLine(
        pt_bins_yield[0],
        1.0,
        pt_bins_yield[-1],
        1.0
    ))
    line_one_anti.SetLineStyle(2)
    line_one_anti.Draw()

    # --------------------------------------------------------
    # Pad 3: Lambda / K0S
    # --------------------------------------------------------
    pdf_canvas.cd(3)
    root.gPad.SetLeftMargin(0.14)
    root.gPad.SetBottomMargin(0.13)
    root.gPad.SetRightMargin(0.04)
    root.gPad.SetTopMargin(0.10)

    g_ratio_k0s = graphs["lambda_over_k0s"]
    g_ratio_k0s.SetMarkerColor(root.kBlack)
    g_ratio_k0s.SetLineColor(root.kBlack)

    valid_lk = [
        row["lambda_over_k0s"]
        for row in rows
        if math.isfinite(row["lambda_over_k0s"])
    ]
    lk_max = max(valid_lk) if valid_lk else 1.0

    frame_lk = keep(root.TH1F(
        f"frame_lambda_k0s_ratio_{cut_index}",
        "",
        100,
        pt_bins_yield[0],
        pt_bins_yield[-1]
    ))
    frame_lk.SetDirectory(0)
    frame_lk.SetMinimum(0.0)
    frame_lk.SetMaximum(max(1.0, 1.4 * lk_max))

    lambda_k0s_title = (
        "(#Lambda+#bar{#Lambda})/(2K^{0}_{S})"
        if lambda_over_k0s_species == "LambdaPlusAntiLambda"
        else "#bar{#Lambda}/K^{0}_{S}" if lambda_over_k0s_species == "AntiLambda"
        else "#Lambda/K^{0}_{S}"
    )

    Format_hist_loc(
        frame_lk,
        "",
        "p_{T} [GeV/c]",
        lambda_k0s_title,
        center_title=True,
        x_title_offset=1.0,
        y_title_offset=1.05
    )
    frame_lk.Draw()
    g_ratio_k0s.Draw("P SAME")

    # --------------------------------------------------------
    # Pad 4: cut and fit status
    # --------------------------------------------------------
    pdf_canvas.cd(4)
    root.gPad.SetLeftMargin(0.05)
    root.gPad.SetBottomMargin(0.05)
    root.gPad.SetRightMargin(0.05)
    root.gPad.SetTopMargin(0.05)

    text = keep(root.TLatex())
    text.SetNDC()
    text.SetTextFont(42)
    text.SetTextSize(0.040)

    text.DrawLatex(0.05, 0.92, cut_name)
    text.SetTextSize(0.030)

    cut_description = cut_lines.get(cut_name, "")
    text.DrawLatex(0.05, 0.84, cut_description)

    y_text = 0.74
    for row in rows:
        status_text = (
            f"{row['ptmin']:.1f}<p_{{T}}<{row['ptmax']:.1f}: "
            f"status K0S/L/#bar{{L}} = "
            f"{row['k0s_fit_status']}/"
            f"{row['lambda_fit_status']}/"
            f"{row['antilambda_fit_status']}"
        )
        text.DrawLatex(0.05, y_text, status_text)
        y_text -= 0.085

    text.DrawLatex(
        0.05,
        0.12,
        "Fit status is for the diagnostic Gaussian only; yield comes from sideband counting."
    )

    pdf_canvas.Update()
    pdf_canvas.Print(yield_ratio_pdf)

pdf_canvas.Print(pdf_close)
pdf_canvas.Draw()

print("Saved PDF :", yield_ratio_pdf)


Saved PDF : output/v0_ratio/V0_yields_and_ratios_all_cuts.pdf


## Reference comparisons retained

The remaining cells are the original comparison/output section. They retain the CMS/STAR overlays, efficiency/BR-corrected ratio plots, corrected K0S and Lambda spectra, Tsallis curves, and mT-scaled reference comparisons.

The external CMS/STAR baryon/meson references correspond to **(Lambda + anti-Lambda)/(2 K0S)**, so those dedicated reference plots intentionally use `average_lambda_over_k0s`. The generic `lambda_over_k0s_species` switch controls the separate selected-ratio quantity.

In [17]:
# ============================================================
# Compare selected cuts on one two-pad canvas
#
# Pad 1: anti-Lambda / Lambda
# Pad 2: (Lambda + anti-Lambda) / (2 K0S)
# ============================================================

selected_cuts_to_plot = [
    "cut02_preselection",
    "cut03_baseline",
    "cut04_pairDCA_15mm",
    "cut05_pairDCA_10mm",
    "cut06_pairDCA_7mm",
    "cut07_pairDCA_5mm",
]

comparison_pdf = (
    outdir
    + "/V0_selected_cuts_antilambda_lambda_and_average_lambda_k0s.pdf"
)

comparison_root = (
    outdir
    + "/V0_selected_cuts_antilambda_lambda_and_average_lambda_k0s.root"
)


In [18]:

# ROOT colors and marker styles for the selected cuts
comparison_colors = [
    root.kBlack,
    root.kBlue + 1,
    root.kRed + 1,
    root.kGreen + 2,
    root.kMagenta + 1,
    root.kOrange + 7,
    root.kCyan + 2,
]

comparison_markers = [
    20,
    21,
    22,
    23,
    24,
    25,
    26,
]

comparison_graphs = {}

for cut_index, cut_name in enumerate(selected_cuts_to_plot):

    if cut_name not in all_cut_results:
        print(f"Warning: {cut_name} is not available in all_cut_results")
        continue

    rows = all_cut_results[cut_name]

    x_values = []
    x_errors = []

    anti_over_lambda_values = []
    anti_over_lambda_errors = []

    average_lambda_over_k0s_values = []
    average_lambda_over_k0s_errors = []

    for row in rows:

        x_values.append(row["pt_center"])
        x_errors.append(row["pt_half_width"])

        # ----------------------------------------------------
        # anti-Lambda / Lambda
        # ----------------------------------------------------
        anti_over_lambda_values.append(
            row["antilambda_over_lambda"]
        )

        anti_over_lambda_errors.append(
            row["antilambda_over_lambda_error"]
        )

        # ----------------------------------------------------
        # (Lambda + anti-Lambda) / (2 K0S)
        # ----------------------------------------------------
        lambda_yield = row["lambda_yield"]
        lambda_error = row["lambda_yield_error"]

        antilambda_yield = row["antilambda_yield"]
        antilambda_error = row["antilambda_yield_error"]

        k0s_yield = row["k0s_yield"]
        k0s_error = row["k0s_yield_error"]

        numerator = lambda_yield + antilambda_yield
        numerator_error = math.sqrt(
            lambda_error**2
            + antilambda_error**2
        )

        denominator = 2.0 * k0s_yield
        denominator_error = 2.0 * k0s_error

        average_ratio, average_ratio_error = ratio_with_error(
            numerator,
            numerator_error,
            denominator,
            denominator_error
        )

        average_lambda_over_k0s_values.append(
            average_ratio
        )

        average_lambda_over_k0s_errors.append(
            average_ratio_error
        )

    # Convert Python lists to C-style double arrays for ROOT
    x_array = array("d", x_values)
    ex_array = array("d", x_errors)

    anti_ratio_array = array(
        "d",
        anti_over_lambda_values
    )
    anti_ratio_error_array = array(
        "d",
        anti_over_lambda_errors
    )

    average_ratio_array = array(
        "d",
        average_lambda_over_k0s_values
    )
    average_ratio_error_array = array(
        "d",
        average_lambda_over_k0s_errors
    )

    graph_anti_ratio = keep(root.TGraphErrors(
        len(x_values),
        x_array,
        anti_ratio_array,
        ex_array,
        anti_ratio_error_array
    ))

    graph_anti_ratio.SetName(
        f"g_antilambda_over_lambda_{cut_name}_comparison"
    )

    graph_average_ratio = keep(root.TGraphErrors(
        len(x_values),
        x_array,
        average_ratio_array,
        ex_array,
        average_ratio_error_array
    ))

    graph_average_ratio.SetName(
        f"g_average_lambda_over_k0s_{cut_name}_comparison"
    )

    color = comparison_colors[
        cut_index % len(comparison_colors)
    ]

    marker = comparison_markers[
        cut_index % len(comparison_markers)
    ]

    for graph in [
        graph_anti_ratio,
        graph_average_ratio,
    ]:
        graph.SetMarkerColor(color)
        graph.SetLineColor(color)
        graph.SetMarkerStyle(marker)
        graph.SetMarkerSize(1.2)
        graph.SetLineWidth(2)

    comparison_graphs[cut_name] = {
        "antilambda_over_lambda": graph_anti_ratio,
        "average_lambda_over_k0s": graph_average_ratio,
    }


# ============================================================
# Find common y-axis ranges over all selected cuts
# ============================================================

all_anti_ratio_values = []
all_average_ratio_values = []

for cut_name in comparison_graphs:

    for row in all_cut_results[cut_name]:

        value_anti = row["antilambda_over_lambda"]

        if math.isfinite(value_anti):
            all_anti_ratio_values.append(value_anti)

        lambda_yield = row["lambda_yield"]
        antilambda_yield = row["antilambda_yield"]
        k0s_yield = row["k0s_yield"]

        if k0s_yield > 0.0:
            value_average = (
                lambda_yield + antilambda_yield
            ) / (2.0 * k0s_yield)

            if math.isfinite(value_average):
                all_average_ratio_values.append(value_average)


anti_ratio_max = (
    max(all_anti_ratio_values)
    if all_anti_ratio_values
    else 1.0
)

average_ratio_max = (
    max(all_average_ratio_values)
    if all_average_ratio_values
    else 1.0
)


# ============================================================
# Draw comparison canvas
# ============================================================

comparison_canvas = keep(root.TCanvas(
    "c_v0_selected_cut_ratios",
    "c_v0_selected_cut_ratios",
    1600,
    700
))

comparison_canvas.Divide(2, 1)


# ------------------------------------------------------------
# Pad 1: anti-Lambda / Lambda
# ------------------------------------------------------------

comparison_canvas.cd(1)

root.gPad.SetLeftMargin(0.14)
root.gPad.SetBottomMargin(0.14)
root.gPad.SetRightMargin(0.04)
root.gPad.SetTopMargin(0.08)

frame_anti_comparison = keep(root.TH1F(
    "frame_anti_comparison",
    "",
    100,
    pt_bins_yield[0],
    pt_bins_yield[-1]
))

frame_anti_comparison.SetDirectory(0)
frame_anti_comparison.SetMinimum(0.0)
frame_anti_comparison.SetMaximum(
    max(1.5, 1.35 * anti_ratio_max)
)

Format_hist_loc(
    frame_anti_comparison,
    "",
    "p_{T} [GeV/c]",
    "#bar{#Lambda}/#Lambda",
    center_title=True,
    x_title_offset=1.0,
    y_title_offset=1.05
)

frame_anti_comparison.Draw()

line_anti_one = keep(root.TLine(
    pt_bins_yield[0],
    1.0,
    pt_bins_yield[-1],
    1.0
))

line_anti_one.SetLineStyle(2)
line_anti_one.SetLineWidth(2)
line_anti_one.Draw()

legend_anti_comparison = keep(root.TLegend(
    0.48,
    0.56,
    0.92,
    0.89
))

legend_anti_comparison.SetBorderSize(0)
legend_anti_comparison.SetFillStyle(0)
legend_anti_comparison.SetTextSize(0.031)

for cut_name in selected_cuts_to_plot:

    if cut_name not in comparison_graphs:
        continue

    graph = comparison_graphs[cut_name][
        "antilambda_over_lambda"
    ]

    graph.Draw("P SAME")

    legend_anti_comparison.AddEntry(
        graph,
        cut_name,
        "lp"
    )

legend_anti_comparison.Draw()

latex_anti = keep(root.TLatex())
latex_anti.SetNDC()
latex_anti.SetTextFont(42)
latex_anti.SetTextSize(0.042)

latex_anti.DrawLatex(
    0.16,
    0.93,
    data_set + ", " + run_text
)


# ------------------------------------------------------------
# Pad 2: (Lambda + anti-Lambda) / (2 K0S)
# ------------------------------------------------------------

comparison_canvas.cd(2)

root.gPad.SetLeftMargin(0.14)
root.gPad.SetBottomMargin(0.14)
root.gPad.SetRightMargin(0.04)
root.gPad.SetTopMargin(0.08)

frame_average_comparison = keep(root.TH1F(
    "frame_average_comparison",
    "",
    100,
    pt_bins_yield[0],
    pt_bins_yield[-1]
))

frame_average_comparison.SetDirectory(0)
frame_average_comparison.SetMinimum(0.0)
frame_average_comparison.SetMaximum(
    max(1.0, 1.35 * average_ratio_max)
)

Format_hist_loc(
    frame_average_comparison,
    "",
    "p_{T} [GeV/c]",
    "(#Lambda+#bar{#Lambda})/(2K^{0}_{S})",
    center_title=True,
    x_title_offset=1.0,
    y_title_offset=1.10
)

frame_average_comparison.Draw()

legend_average_comparison = keep(root.TLegend(
    0.48,
    0.56,
    0.92,
    0.89
))

legend_average_comparison.SetBorderSize(0)
legend_average_comparison.SetFillStyle(0)
legend_average_comparison.SetTextSize(0.031)

for cut_name in selected_cuts_to_plot:

    if cut_name not in comparison_graphs:
        continue

    graph = comparison_graphs[cut_name][
        "average_lambda_over_k0s"
    ]

    graph.Draw("P SAME")

    legend_average_comparison.AddEntry(
        graph,
        cut_name,
        "lp"
    )

legend_average_comparison.Draw()

latex_average = keep(root.TLatex())
latex_average.SetNDC()
latex_average.SetTextFont(42)
latex_average.SetTextSize(0.06)

latex_average.DrawLatex(
    0.16,
    0.93,
    data_set + ", " + run_text
)


# ============================================================
# Save canvas
# ============================================================

comparison_canvas.Update()
comparison_canvas.SaveAs("output/v0_ratio/comparison_selected_cuts1.pdf")
comparison_canvas.Draw()

print("Saved comparison PDF:", comparison_pdf)


# ============================================================
# Save comparison graphs to ROOT
# ============================================================

comparison_output = root.TFile.Open(
    comparison_root,
    "RECREATE"
)

for cut_name, graph_dict in comparison_graphs.items():

    comparison_output.mkdir(cut_name)
    comparison_output.cd(cut_name)

    graph_dict["antilambda_over_lambda"].Write()
    graph_dict["average_lambda_over_k0s"].Write()

comparison_output.Close()

print("Saved comparison ROOT file:", comparison_root)

Saved comparison PDF: output/v0_ratio/V0_selected_cuts_antilambda_lambda_and_average_lambda_k0s.pdf
Saved comparison ROOT file: output/v0_ratio/V0_selected_cuts_antilambda_lambda_and_average_lambda_k0s.root


In [19]:
import ROOT
import numpy as np

ROOT.gStyle.SetOptStat(0)

# ============================================================
# Digitized points
# ============================================================

# First plot: black, uncorrected Lambda0/K0S points
pt_black = np.array([
    0.75, 0.85, 0.95, 1.05, 1.15, 1.25, 1.35,
    1.45, 1.65, 1.95, 2.25, 2.55, 2.85, 3.50
], dtype="float64")

ratio_black = np.array([
    0.0588, 0.0716, 0.0672, 0.0681, 0.0737, 0.0805, 0.0849,
    0.0912, 0.1004, 0.1107, 0.1122, 0.0866, 0.0565, 0.0777
], dtype="float64")


# Second plot: CMS red points with systematic uncertainties
pt_cms = np.array([
    0.50, 0.70, 0.90, 1.10, 1.30,
    1.50, 1.70, 1.90, 2.10, 2.30,
    2.50, 2.70, 2.90, 3.20, 3.60
], dtype="float64")

ratio_cms = np.array([
    0.239, 0.296, 0.370, 0.408, 0.466,
    0.513, 0.541, 0.548, 0.536, 0.548,
    0.548, 0.511, 0.541, 0.492, 0.439
], dtype="float64")

cms_syst_low = np.array([
    0.020, 0.027, 0.029, 0.033, 0.038,
    0.031, 0.034, 0.034, 0.035, 0.034,
    0.034, 0.033, 0.034, 0.032, 0.028
], dtype="float64")

cms_syst_high = np.array([
    0.020, 0.027, 0.029, 0.033, 0.038,
    0.031, 0.034, 0.034, 0.035, 0.034,
    0.034, 0.033, 0.034, 0.032, 0.028
], dtype="float64")

# Horizontal systematic-box half-widths estimated from the CMS pT binning
cms_pt_width = np.array([
    0.10, 0.10, 0.10, 0.10, 0.10,
    0.10, 0.10, 0.10, 0.10, 0.10,
    0.10, 0.10, 0.10, 0.15, 0.20
], dtype="float64")


# ============================================================
# Create the two ROOT graphs
# ============================================================

# Graph 1: black points from the first plot
g_black = ROOT.TGraph(
    len(pt_black),
    pt_black,
    ratio_black
)

g_black.SetName("g_black_uncorrected")
g_black.SetTitle("Digitized black points")
g_black.SetMarkerStyle(20)
g_black.SetMarkerSize(1.1)
g_black.SetMarkerColor(ROOT.kBlack)
g_black.SetLineColor(ROOT.kBlack)


# Graph 2: CMS points with systematic uncertainties
g_cms = ROOT.TGraphAsymmErrors(
    len(pt_cms),
    pt_cms,
    ratio_cms,
    cms_pt_width,
    cms_pt_width,
    cms_syst_low,
    cms_syst_high
)

g_cms.SetName("g_cms_systematic")
g_cms.SetTitle("Digitized CMS points with systematic uncertainties")
g_cms.SetMarkerStyle(20)
g_cms.SetMarkerSize(1.1)
g_cms.SetMarkerColor(ROOT.kRed + 1)
g_cms.SetLineColor(ROOT.kRed + 1)
g_cms.SetFillColorAlpha(ROOT.kRed, 0.20)
g_cms.SetFillStyle(1001)


# Keep references alive in a notebook
digitized_graphs = [g_black, g_cms]

In [20]:
# ROOT colors and marker styles for the selected cuts
comparison_colors = [
    root.kBlack,
    root.kBlue + 1,
    root.kRed + 1,
    root.kGreen + 2,
    root.kMagenta + 1,
    root.kOrange + 7,
    root.kCyan + 2,
]

comparison_markers = [
    20,
    21,
    22,
    23,
    24,
    25,
    26,
]

# Different line styles identify the particle species
k0s_line_style = 1
lambda_line_style = 2
antilambda_line_style = 3

comparison_graphs = {}

for cut_index, cut_name in enumerate(selected_cuts_to_plot):

    if cut_name not in all_cut_results:
        print(
            f"Warning: {cut_name} is not available "
            "in all_cut_results"
        )
        continue

    rows = all_cut_results[cut_name]

    x_values = []
    x_errors = []

    k0s_yield_values = []
    k0s_yield_errors = []

    lambda_yield_values = []
    lambda_yield_errors = []

    antilambda_yield_values = []
    antilambda_yield_errors = []

    anti_over_lambda_values = []
    anti_over_lambda_errors = []

    average_lambda_over_k0s_values = []
    average_lambda_over_k0s_errors = []

    for row in rows:

        x_values.append(row["pt_center"])
        x_errors.append(row["pt_half_width"])

        # ----------------------------------------------------
        # Individual yields
        # ----------------------------------------------------
        k0s_yield_values.append(
            row["k0s_yield"]
        )
        k0s_yield_errors.append(
            row["k0s_yield_error"]
        )

        lambda_yield_values.append(
            row["lambda_yield"]
        )
        lambda_yield_errors.append(
            row["lambda_yield_error"]
        )

        antilambda_yield_values.append(
            row["antilambda_yield"]
        )
        antilambda_yield_errors.append(
            row["antilambda_yield_error"]
        )

        # ----------------------------------------------------
        # anti-Lambda / Lambda
        # ----------------------------------------------------
        anti_over_lambda_values.append(
            row["antilambda_over_lambda"]
        )

        anti_over_lambda_errors.append(
            row["antilambda_over_lambda_error"]
        )

        # ----------------------------------------------------
        # (Lambda + anti-Lambda) / (2 K0S)
        # ----------------------------------------------------
        lambda_yield = row["lambda_yield"]
        lambda_error = row["lambda_yield_error"]

        antilambda_yield = row["antilambda_yield"]
        antilambda_error = row[
            "antilambda_yield_error"
        ]

        k0s_yield = row["k0s_yield"]
        k0s_error = row["k0s_yield_error"]

        numerator = (
            lambda_yield
            + antilambda_yield
        )

        numerator_error = math.sqrt(
            lambda_error**2
            + antilambda_error**2
        )

        denominator = 2.0 * k0s_yield
        denominator_error = 2.0 * k0s_error

        average_ratio, average_ratio_error = (
            ratio_with_error(
                numerator,
                numerator_error,
                denominator,
                denominator_error
            )
        )

        average_lambda_over_k0s_values.append(
            average_ratio
        )

        average_lambda_over_k0s_errors.append(
            average_ratio_error
        )

    # ========================================================
    # Convert to C-style arrays
    # ========================================================

    x_array = array("d", x_values)
    ex_array = array("d", x_errors)

    k0s_yield_array = array(
        "d",
        k0s_yield_values
    )
    k0s_yield_error_array = array(
        "d",
        k0s_yield_errors
    )

    lambda_yield_array = array(
        "d",
        lambda_yield_values
    )
    lambda_yield_error_array = array(
        "d",
        lambda_yield_errors
    )

    antilambda_yield_array = array(
        "d",
        antilambda_yield_values
    )
    antilambda_yield_error_array = array(
        "d",
        antilambda_yield_errors
    )

    anti_ratio_array = array(
        "d",
        anti_over_lambda_values
    )
    anti_ratio_error_array = array(
        "d",
        anti_over_lambda_errors
    )

    average_ratio_array = array(
        "d",
        average_lambda_over_k0s_values
    )
    average_ratio_error_array = array(
        "d",
        average_lambda_over_k0s_errors
    )

    # ========================================================
    # Yield graphs
    # ========================================================

    graph_k0s_yield = keep(root.TGraphErrors(
        len(x_values),
        x_array,
        k0s_yield_array,
        ex_array,
        k0s_yield_error_array
    ))

    graph_k0s_yield.SetName(
        f"g_k0s_yield_{cut_name}_comparison"
    )

    graph_lambda_yield = keep(root.TGraphErrors(
        len(x_values),
        x_array,
        lambda_yield_array,
        ex_array,
        lambda_yield_error_array
    ))

    graph_lambda_yield.SetName(
        f"g_lambda_yield_{cut_name}_comparison"
    )

    graph_antilambda_yield = keep(
        root.TGraphErrors(
            len(x_values),
            x_array,
            antilambda_yield_array,
            ex_array,
            antilambda_yield_error_array
        )
    )

    graph_antilambda_yield.SetName(
        f"g_antilambda_yield_"
        f"{cut_name}_comparison"
    )

    # ========================================================
    # Ratio graphs
    # ========================================================

    graph_anti_ratio = keep(root.TGraphErrors(
        len(x_values),
        x_array,
        anti_ratio_array,
        ex_array,
        anti_ratio_error_array
    ))

    graph_anti_ratio.SetName(
        f"g_antilambda_over_lambda_"
        f"{cut_name}_comparison"
    )

    graph_average_ratio = keep(
        root.TGraphErrors(
            len(x_values),
            x_array,
            average_ratio_array,
            ex_array,
            average_ratio_error_array
        )
    )

    graph_average_ratio.SetName(
        f"g_average_lambda_over_k0s_"
        f"{cut_name}_comparison"
    )

    color = comparison_colors[
        cut_index % len(comparison_colors)
    ]

    marker = comparison_markers[
        cut_index % len(comparison_markers)
    ]

    # Same color and marker for one cut
    for graph in [
        graph_k0s_yield,
        graph_lambda_yield,
        graph_antilambda_yield,
        graph_anti_ratio,
        graph_average_ratio,
    ]:
        graph.SetMarkerColor(color)
        graph.SetLineColor(color)
        graph.SetMarkerStyle(marker)
        graph.SetMarkerSize(1.05)
        graph.SetLineWidth(2)

    # Particle species are distinguished by line style
    graph_k0s_yield.SetLineStyle(
        k0s_line_style
    )
    graph_lambda_yield.SetLineStyle(
        lambda_line_style
    )
    graph_antilambda_yield.SetLineStyle(
        antilambda_line_style
    )

    comparison_graphs[cut_name] = {
        "k0s_yield": graph_k0s_yield,
        "lambda_yield": graph_lambda_yield,
        "antilambda_yield":
            graph_antilambda_yield,
        "antilambda_over_lambda":
            graph_anti_ratio,
        "average_lambda_over_k0s":
            graph_average_ratio,
    }

In [21]:
# ============================================================
# Determine common plotting ranges
# ============================================================

all_yield_values = []
all_anti_ratio_values = []
all_average_ratio_values = []

for cut_name in comparison_graphs:

    for row in all_cut_results[cut_name]:

        # ----------------------------------------------------
        # Yields
        # ----------------------------------------------------
        for key in [
            "k0s_yield",
            "lambda_yield",
            "antilambda_yield",
        ]:
            value = row[key]

            if (
                math.isfinite(value)
                and value > 0.0
            ):
                all_yield_values.append(value)

        # ----------------------------------------------------
        # anti-Lambda / Lambda
        # ----------------------------------------------------
        value_anti = row[
            "antilambda_over_lambda"
        ]

        if math.isfinite(value_anti):
            all_anti_ratio_values.append(
                value_anti
            )

        # ----------------------------------------------------
        # Average Lambda / K0S
        # ----------------------------------------------------
        lambda_yield = row["lambda_yield"]
        antilambda_yield = row[
            "antilambda_yield"
        ]
        k0s_yield = row["k0s_yield"]

        if k0s_yield > 0.0:

            value_average = (
                lambda_yield
                + antilambda_yield
            ) / (2.0 * k0s_yield)

            if math.isfinite(value_average):
                all_average_ratio_values.append(
                    value_average
                )


yield_min = (
    min(all_yield_values)
    if all_yield_values
    else 1.0
)

yield_max = (
    max(all_yield_values)
    if all_yield_values
    else 10.0
)

anti_ratio_max = (
    max(all_anti_ratio_values)
    if all_anti_ratio_values
    else 1.0
)

average_ratio_max = (
    max(all_average_ratio_values)
    if all_average_ratio_values
    else 1.0
)

In [22]:
# ============================================================
# Prepare detailed cut descriptions from cut_lines
# ============================================================

def get_cut_description_lines(cut_name):
    """
    Return cut_lines[cut_name] as a clean list of strings.

    Supported forms:
        cut_lines[cut_name] = ["p_{T}>0.2", "pair DCA < 1 cm"]
        cut_lines[cut_name] = "p_{T}>0.2; pair DCA < 1 cm"
    """

    if cut_name not in cut_lines:
        return [cut_name]

    description = cut_lines[cut_name]

    if isinstance(description, str):
        # Split a single string into separate selections
        description = description.replace("\n", ";")
        lines = [
            line.strip()
            for line in description.split(";")
            if line.strip()
        ]
    else:
        lines = [
            str(line).strip()
            for line in description
            if str(line).strip()
        ]

    return lines


# Only cuts that actually have graphs
valid_selected_cuts = [
    cut_name
    for cut_name in selected_cuts_to_plot
    if cut_name in comparison_graphs
]

cut_description_lines = {
    cut_name: get_cut_description_lines(cut_name)
    for cut_name in valid_selected_cuts
}


# ============================================================
# Find selections common to all selected cuts
# ============================================================

common_cut_lines = []

if valid_selected_cuts:

    first_cut_name = valid_selected_cuts[0]

    for line in cut_description_lines[first_cut_name]:

        if all(
            line in cut_description_lines[other_cut_name]
            for other_cut_name in valid_selected_cuts[1:]
        ):
            common_cut_lines.append(line)


# Remove common selections from the individual descriptions
cut_specific_lines = {}

for cut_name in valid_selected_cuts:

    cut_specific_lines[cut_name] = [
        line
        for line in cut_description_lines[cut_name]
        if line not in common_cut_lines
    ]


print("Common selections:")
for line in common_cut_lines:
    print("   ", line)

print("\nCut-dependent selections:")
for cut_name in valid_selected_cuts:
    print(cut_name)
    for line in cut_specific_lines[cut_name]:
        print("   ", line)

#adding pt>0.2 Gev to common_cut_lines
common_cut_lines.append("p_{T} > 0.2 GeV/c, Lproj>2.0 cm, Sailors cut: #phi^{+}-#phi^{-} > #phi_{cut}")

Common selections:

Cut-dependent selections:
cut02_preselection
    |z_{vtx}|<15 cm, |PCA_{z}|<0.7 cm, |PCA_{xy}|<3.0 cm, DIRA>0.80
cut03_baseline
    |z_{vtx}|<15 cm, |PCA_{z}|<0.5 cm, |PCA_{xy}|<2.0 cm, DIRA>0.85
cut04_pairDCA_15mm
    |z_{vtx}|<12 cm, |PCA_{z}|<0.5 cm, |PCA_{xy}|<1.5 cm, DIRA>0.88
cut05_pairDCA_10mm
    |z_{vtx}|<10 cm, |PCA_{z}|<0.4 cm, |PCA_{xy}|<1.0 cm, DIRA>0.90
cut06_pairDCA_7mm
    |z_{vtx}|<10 cm, |PCA_{z}|<0.3 cm, |PCA_{xy}|<0.7 cm, DIRA>0.93
cut07_pairDCA_5mm
    |z_{vtx}|<10 cm, |PCA_{z}|<0.2 cm, |PCA_{xy}|<0.5 cm, DIRA>0.95


In [23]:
# ============================================================
# Draw four-pad comparison canvas
# ============================================================

comparison_canvas = keep(root.TCanvas(
    "c_v0_selected_cut_ratios",
    "c_v0_selected_cut_ratios",
    1600,
    1300
))

comparison_canvas.Divide(2, 2)


# ============================================================
# Pad 1: K0S, Lambda, and anti-Lambda yields
# ============================================================

comparison_canvas.cd(1)

root.gPad.SetLogy()
root.gPad.SetLeftMargin(0.14)
root.gPad.SetBottomMargin(0.14)
root.gPad.SetRightMargin(0.04)
root.gPad.SetTopMargin(0.09)

frame_yields_comparison = keep(root.TH1F(
    "frame_yields_comparison",
    "",
    100,
    pt_bins_yield[0],
    pt_bins_yield[-1]
))

frame_yields_comparison.SetDirectory(0)

frame_yields_comparison.SetMinimum(
    max(
        1.0e-6,
        0.25 * yield_min
    )
)

frame_yields_comparison.SetMaximum(
    5.0 * yield_max
)

Format_hist_loc(
    frame_yields_comparison,
    "",
    "p_{T} [GeV/c]",
    "Raw candidate yield",
    center_title=True,
    x_title_offset=1.0,
    y_title_offset=1.10
)

frame_yields_comparison.Draw()

for cut_name in selected_cuts_to_plot:

    if cut_name not in comparison_graphs:
        continue

    graph_dict = comparison_graphs[cut_name]

    graph_dict["k0s_yield"].Draw(
        "P SAME"
    )

    graph_dict["lambda_yield"].Draw(
        "P SAME"
    )

    graph_dict["antilambda_yield"].Draw(
        "P SAME"
    )

latex_yields = keep(root.TLatex())
latex_yields.SetNDC()
latex_yields.SetTextFont(42)
latex_yields.SetTextSize(0.060)

latex_yields.DrawLatex(
    0.16,
    0.94,
    data_set + ", " + run_text
)


# ============================================================
# Pad 2: detailed cut descriptions
# ============================================================

comparison_canvas.cd(2)

root.gPad.SetLeftMargin(0.03)
root.gPad.SetBottomMargin(0.03)
root.gPad.SetRightMargin(0.03)
root.gPad.SetTopMargin(0.03)
root.gPad.SetTicks(0, 0)

legend_frame = keep(root.TH1F(
    "legend_frame",
    "",
    1,
    0.0,
    1.0
))

legend_frame.SetDirectory(0)
legend_frame.SetMinimum(0.0)
legend_frame.SetMaximum(1.0)

legend_frame.GetXaxis().SetLabelSize(0.0)
legend_frame.GetYaxis().SetLabelSize(0.0)
legend_frame.GetXaxis().SetTickLength(0.0)
legend_frame.GetYaxis().SetTickLength(0.0)

legend_frame.Draw("AXIS")


latex_legend = keep(root.TLatex())
latex_legend.SetNDC()
latex_legend.SetTextFont(42)


# ------------------------------------------------------------
# Common selections
# ------------------------------------------------------------

current_y = 0.93

latex_legend.SetTextFont(62)
latex_legend.SetTextSize(0.050)
latex_legend.DrawLatex(
    0.06,
    current_y,
    "Common selections"
)

current_y -= 0.050

latex_legend.SetTextFont(42)
latex_legend.SetTextSize(0.047)

for line in common_cut_lines:
    latex_legend.DrawLatex(
        0.08,
        current_y,
        "#bullet  " + line
    )
    current_y -= 0.034


separator = keep(root.TLine(
    0.05,
    current_y - 0.005,
    0.95,
    current_y - 0.005
))

separator.SetNDC()
separator.SetLineStyle(3)
separator.SetLineColor(root.kGray + 1)
separator.Draw()

current_y -= 0.050


# ------------------------------------------------------------
# Cut-dependent selections
# ------------------------------------------------------------

latex_legend.SetTextFont(62)
latex_legend.SetTextSize(0.050)
latex_legend.DrawLatex(
    0.06,
    current_y,
    "Cut-dependent selections"
)

current_y -= 0.052

cut_legend_markers = {}

# Reduce text size automatically if many cuts are drawn
cut_text_size = (
    0.027
    if len(valid_selected_cuts) <= 4
    else 0.044
)

line_spacing = (
    0.034
    if len(valid_selected_cuts) <= 4
    else 0.044
)

for cut_name in valid_selected_cuts:

    graph = comparison_graphs[cut_name][
        "antilambda_over_lambda"
    ]

    color = graph.GetMarkerColor()
    marker_style = graph.GetMarkerStyle()

    # Small marker representing this cut
    marker_graph = keep(root.TGraph(1))
    marker_graph.SetPoint(
        0,
        0.045,
        current_y
    )

    marker_graph.SetMarkerColor(color)
    marker_graph.SetLineColor(color)
    marker_graph.SetMarkerStyle(marker_style)
    marker_graph.SetMarkerSize(1.25)
    marker_graph.Draw("P SAME")

    cut_legend_markers[cut_name] = marker_graph

    # Cut name
    latex_legend.SetTextFont(62)
    latex_legend.SetTextSize(cut_text_size + 0.002)
    latex_legend.SetTextColor(color)

    latex_legend.DrawLatex(
        0.105,
        current_y,
        cut_name
    )

    latex_legend.SetTextColor(root.kBlack)

    current_y -= line_spacing

    # Detailed cut-dependent lines from cut_lines
    latex_legend.SetTextFont(42)
    latex_legend.SetTextSize(cut_text_size)

    specific_lines = cut_specific_lines[cut_name]

    if specific_lines:
        for line in specific_lines:
            latex_legend.DrawLatex(
                0.085,
                current_y,
                "  " + line
            )
            current_y -= line_spacing
    else:
        latex_legend.SetTextColor(root.kGray + 2)

        latex_legend.DrawLatex(
            0.105,
            current_y,
            "Same as common selection"
        )

        latex_legend.SetTextColor(root.kBlack)
        current_y -= line_spacing

    current_y -= 0.012


# ------------------------------------------------------------
# Particle-species line styles
# ------------------------------------------------------------

species_separator = keep(root.TLine(
    0.05,
    max(0.10, current_y),
    0.95,
    max(0.10, current_y)
))

species_separator.SetNDC()
species_separator.SetLineStyle(3)
species_separator.SetLineColor(root.kGray + 1)
species_separator.Draw()

species_y = max(0.055, current_y - 0.050)

species_k0s = keep(root.TLine(
    0.08,
    species_y,
    0.18,
    species_y
))

species_lambda = keep(root.TLine(
    0.38,
    species_y,
    0.48,
    species_y
))

species_antilambda = keep(root.TLine(
    0.68,
    species_y,
    0.78,
    species_y
))

for line in [
    species_k0s,
    species_lambda,
    species_antilambda,
]:
    line.SetNDC()
    line.SetLineColor(root.kBlack)
    line.SetLineWidth(3)

species_k0s.SetLineStyle(k0s_line_style)
species_lambda.SetLineStyle(lambda_line_style)
species_antilambda.SetLineStyle(
    antilambda_line_style
)

species_k0s.Draw()
species_lambda.Draw()
species_antilambda.Draw()

latex_legend.SetTextFont(42)
latex_legend.SetTextSize(0.057)

latex_legend.DrawLatex(
    0.20,
    species_y,
    "K^{0}_{S}"
)

latex_legend.DrawLatex(
    0.50,
    species_y,
    "#Lambda"
)

latex_legend.DrawLatex(
    0.80,
    species_y,
    "#bar{#Lambda}"
)
# ============================================================
# Pad 3: anti-Lambda / Lambda
# ============================================================

comparison_canvas.cd(3)

root.gPad.SetLeftMargin(0.14)
root.gPad.SetBottomMargin(0.14)
root.gPad.SetRightMargin(0.04)
root.gPad.SetTopMargin(0.08)

frame_anti_comparison = keep(root.TH1F(
    "frame_anti_comparison",
    "",
    100,
    pt_bins_yield[0],
    pt_bins_yield[-1]
))

frame_anti_comparison.SetDirectory(0)
frame_anti_comparison.SetMinimum(0.0)

frame_anti_comparison.SetMaximum(
    max(
        1.5,
        1.35 * anti_ratio_max
    )
)

Format_hist_loc(
    frame_anti_comparison,
    "",
    "p_{T} [GeV/c]",
    "#bar{#Lambda}/#Lambda",
    center_title=True,
    x_title_offset=1.0,
    y_title_offset=1.05
)

frame_anti_comparison.Draw()

line_anti_one = keep(root.TLine(
    pt_bins_yield[0],
    0.76,
    pt_bins_yield[-1],
    0.76
))

latex_doted_line = keep(root.TLegend(
    0.16,
    0.76,
    0.36,
    0.86
))
latex_doted_line.SetBorderSize(0)
latex_doted_line.SetFillStyle(0)
latex_doted_line.SetTextFont(42)
latex_doted_line.SetTextSize(0.050)
latex_doted_line.AddEntry(
    line_anti_one,
    "0.76 ratio of #bar{p}/p ratio from RHIC at 200 GeV",
    "l"
)
latex_doted_line.Draw()

line_anti_one.SetLineStyle(2)
line_anti_one.SetLineWidth(2)
line_anti_one.Draw()

for cut_name in selected_cuts_to_plot:

    if cut_name not in comparison_graphs:
        continue

    comparison_graphs[cut_name][
        "antilambda_over_lambda"
    ].Draw("PZ SAME")

latex_anti = keep(root.TLatex())
latex_anti.SetNDC()
latex_anti.SetTextFont(42)
latex_anti.SetTextSize(0.060)

latex_anti.DrawLatex(
    0.16,
    0.93,
    data_set + ", " + run_text
)


# ============================================================
# Pad 4: average Lambda / K0S with external results
# ============================================================

comparison_canvas.cd(4)

root.gPad.SetLeftMargin(0.14)
root.gPad.SetBottomMargin(0.14)
root.gPad.SetRightMargin(0.04)
root.gPad.SetTopMargin(0.08)

# Include external points when selecting the maximum
external_average_max = 0.0

for external_graph in [
    g_black,
    g_cms,
]:
    for point_index in range(
        external_graph.GetN()
    ):
        external_average_max = max(
            external_average_max,
            external_graph.GetPointY(
                point_index
            )
        )

frame_average_comparison = keep(root.TH1F(
    "frame_average_comparison",
    "",
    100,
    pt_bins_yield[0],
    pt_bins_yield[-1]
))

frame_average_comparison.SetDirectory(0)
frame_average_comparison.SetMinimum(0.0)

frame_average_comparison.SetMaximum(
    max(
        0.70,
        1.25 * average_ratio_max,
        1.25 * external_average_max
    )
)

Format_hist_loc(
    frame_average_comparison,
    "",
    "p_{T} [GeV/c]",
    "(#Lambda+#bar{#Lambda})/(2K^{0}_{S})",
    center_title=True,
    x_title_offset=1.0,
    y_title_offset=1.10
)

frame_average_comparison.Draw()

# ------------------------------------------------------------
# CMS systematic uncertainty boxes
# ------------------------------------------------------------

g_cms.SetMarkerStyle(20)
g_cms.SetMarkerSize(0.95)
g_cms.SetMarkerColor(root.kRed + 1)
g_cms.SetLineColor(root.kRed + 1)

g_cms.SetFillColorAlpha(
    root.kRed,
    0.20
)
g_cms.SetFillStyle(1001)

# Draw systematic boxes first
g_cms.Draw("2 SAME")

# Draw central points above boxes
g_cms.Draw("P SAME")


# ------------------------------------------------------------
# sPHENIX PPG16 black points
# ------------------------------------------------------------

g_black.SetMarkerStyle(24)
g_black.SetMarkerSize(1.05)
g_black.SetMarkerColor(root.kBlack)
g_black.SetLineColor(root.kBlack)
g_black.SetLineWidth(2)

g_black.Draw("P SAME")


# ------------------------------------------------------------
# Current analysis results
# ------------------------------------------------------------

for cut_name in selected_cuts_to_plot:

    if cut_name not in comparison_graphs:
        continue

    comparison_graphs[cut_name][
        "average_lambda_over_k0s"
    ].Draw("PZ SAME")


# ------------------------------------------------------------
# External-data legend
# ------------------------------------------------------------

legend_external = keep(root.TLegend(
    0.15,
    0.72,
    0.52,
    0.92
))

legend_external.SetBorderSize(0)
legend_external.SetFillStyle(0)
legend_external.SetTextFont(42)
legend_external.SetTextSize(0.050)

legend_external.AddEntry(
    g_cms,
    "CMS, p+p #sqrt{s}=7 TeV",
    "pf"
)

legend_external.AddEntry(
    g_black,
    "sPHENIX PPG16",
    "p"
)

legend_external.Draw()

latex_average = keep(root.TLatex())
latex_average.SetNDC()
latex_average.SetTextFont(42)
latex_average.SetTextSize(0.060)

latex_average.DrawLatex(
    0.16,
    0.95,
    data_set + ", " + run_text
)


# ============================================================
# Save canvas
# ============================================================

comparison_canvas.Update()
comparison_canvas.SaveAs(comparison_pdf)
comparison_canvas.Draw()

print(
    "Saved comparison PDF:",
    comparison_pdf
)

Saved comparison PDF: output/v0_ratio/V0_selected_cuts_antilambda_lambda_and_average_lambda_k0s.pdf


In [24]:
# ============================================================
# Save comparison graphs to ROOT
# ============================================================

comparison_output = root.TFile.Open(
    comparison_root,
    "RECREATE"
)

for cut_name, graph_dict in (
    comparison_graphs.items()
):

    comparison_output.mkdir(cut_name)
    comparison_output.cd(cut_name)

    graph_dict["k0s_yield"].Write()
    graph_dict["lambda_yield"].Write()
    graph_dict["antilambda_yield"].Write()

    graph_dict[
        "antilambda_over_lambda"
    ].Write()

    graph_dict[
        "average_lambda_over_k0s"
    ].Write()

comparison_output.cd()

g_cms.Write(
    "g_cms_lambda_over_2k0s_digitized"
)

g_black.Write(
    "g_sphenix_ppg16_lambda_over_k0s_digitized"
)

comparison_canvas.Write()

comparison_output.Close()

print(
    "Saved comparison ROOT file:",
    comparison_root
)

Saved comparison ROOT file: output/v0_ratio/V0_selected_cuts_antilambda_lambda_and_average_lambda_k0s.root


In [25]:
print("All-cut V0 yield and ratio analysis complete.")
print("CSV output :", yield_csv)
print("ROOT output:", yield_root_file)
print("PDF output :", yield_ratio_pdf)


All-cut V0 yield and ratio analysis complete.
CSV output : output/v0_ratio/V0_yields_and_ratios_all_cuts.csv
ROOT output: output/v0_ratio/V0_yields_and_ratios_all_cuts.root
PDF output : output/v0_ratio/V0_yields_and_ratios_all_cuts.pdf


In [26]:
# ============================================================
# Efficiency + branching-ratio correction
# ============================================================
# Uses the full data pT interval, not only the bin center.

br_k0s_to_pipi = 0.6920
br_lambda_to_ppi = 0.6390
br_ratio_k0s_over_lambda = br_k0s_to_pipi / br_lambda_to_ppi


def multiply_ratio_with_factor(raw_ratio, raw_error, factor, factor_error):
    if not all(math.isfinite(v) for v in [raw_ratio, raw_error, factor, factor_error]):
        return float("nan"), float("nan")
    value = raw_ratio * factor
    rel2 = 0.0
    if raw_ratio != 0.0:
        rel2 += (raw_error/raw_ratio)**2
    if factor != 0.0:
        rel2 += (factor_error/factor)**2
    return value, abs(value)*math.sqrt(rel2)


for cut_name, rows in all_cut_results.items():
    for row in rows:
        # Defaults
        for key in [
            "eff_k0s_over_lambda", "eff_k0s_over_lambda_error",
            "lambda_over_k0s_corrected", "lambda_over_k0s_corrected_error",
            "average_lambda_over_k0s_corrected", "average_lambda_over_k0s_corrected_error",
        ]:
            row[key] = float("nan")
        row["eff_ptmin"] = row["ptmin"]
        row["eff_ptmax"] = row["ptmax"]
        row["efficiency_error_complete"] = False
        row["br_k0s_over_lambda"] = br_ratio_k0s_over_lambda

        # Always keep the published-reference average ratio available.
        avg_y = 0.5*(row["lambda_yield"] + row["antilambda_yield"])
        avg_e = 0.5*math.sqrt(row["lambda_yield_error"]**2 + row["antilambda_yield_error"]**2)
        avg_raw, avg_raw_err = ratio_with_error(
            avg_y, avg_e, row["k0s_yield"], row["k0s_yield_error"]
        )
        row["average_lambda_over_k0s"] = avg_raw
        row["average_lambda_over_k0s_error"] = avg_raw_err

        ek = effective_efficiency(k0s_eff_percent, cut_name, row["ptmin"], row["ptmax"])
        el = effective_efficiency(lambda_eff_percent, cut_name, row["ptmin"], row["ptmax"])
        if ek is None or el is None:
            continue

        eff_ratio, eff_ratio_err = ratio_with_error(ek[0], ek[1], el[0], el[1])
        if not math.isfinite(eff_ratio):
            continue

        row["eff_k0s_over_lambda"] = eff_ratio
        row["eff_k0s_over_lambda_error"] = eff_ratio_err
        row["efficiency_error_complete"] = True

        factor = eff_ratio * br_ratio_k0s_over_lambda
        factor_err = abs(factor) * eff_ratio_err/eff_ratio if eff_ratio != 0.0 else float("nan")

        row["lambda_over_k0s_corrected"], row["lambda_over_k0s_corrected_error"] = multiply_ratio_with_factor(
            row["lambda_over_k0s"], row["lambda_over_k0s_error"], factor, factor_err
        )
        row["average_lambda_over_k0s_corrected"], row["average_lambda_over_k0s_corrected_error"] = multiply_ratio_with_factor(
            avg_raw, avg_raw_err, factor, factor_err
        )

print("Efficiency correction uses overlap-weighted efficiencies over each full data pT bin.")


Efficiency correction uses overlap-weighted efficiencies over each full data pT bin.


In [27]:
# ============================================================
# Save the new corrected quantities, while keeping all old files intact
# ============================================================

corrected_csv = outdir + "/V0_efficiency_BR_corrected_ratios.csv"
corrected_root = outdir + "/V0_efficiency_BR_corrected_ratios.root"

corrected_columns = [
    "cut",
    "ptmin",
    "ptmax",
    "pt_center",
    "lambda_over_k0s",
    "lambda_over_k0s_error",
    "average_lambda_over_k0s",
    "average_lambda_over_k0s_error",
    "eff_k0s_over_lambda",
    "eff_k0s_over_lambda_error",
    "eff_ptmin",
    "eff_ptmax",
    "efficiency_error_complete",
    "br_k0s_over_lambda",
    "lambda_over_k0s_corrected",
    "lambda_over_k0s_corrected_error",
    "average_lambda_over_k0s_corrected",
    "average_lambda_over_k0s_corrected_error",
]

with open(corrected_csv, "w", newline="") as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=corrected_columns)
    writer.writeheader()
    for cut_name in cuts:
        for row in all_cut_results[cut_name]:
            writer.writerow({key: row[key] for key in corrected_columns})

corrected_output = root.TFile.Open(corrected_root, "RECREATE")
corrected_graphs_by_cut = {}

for cut_name, rows in all_cut_results.items():
    corrected_output.mkdir(cut_name)
    corrected_output.cd(cut_name)

    # Keep only points for which a correction exists.
    valid_rows = [
        row for row in rows
        if math.isfinite(row["eff_k0s_over_lambda"])
    ]

    if not valid_rows:
        continue

    graphs = {
        "lambda_over_k0s_raw": make_graph(
            f"g_lambda_over_k0s_raw_{cut_name}",
            ";p_{T} [GeV/c];#Lambda/K^{0}_{S}",
            valid_rows,
            "lambda_over_k0s",
            "lambda_over_k0s_error"
        ),
        "lambda_over_k0s_corrected": make_graph(
            f"g_lambda_over_k0s_corrected_{cut_name}",
            ";p_{T} [GeV/c];corrected #Lambda/K^{0}_{S}",
            valid_rows,
            "lambda_over_k0s_corrected",
            "lambda_over_k0s_corrected_error"
        ),
        "average_lambda_over_k0s_raw": make_graph(
            f"g_average_lambda_over_k0s_raw_{cut_name}",
            ";p_{T} [GeV/c];(#Lambda+#bar{#Lambda})/(2K^{0}_{S})",
            valid_rows,
            "average_lambda_over_k0s",
            "average_lambda_over_k0s_error"
        ),
        "average_lambda_over_k0s_corrected": make_graph(
            f"g_average_lambda_over_k0s_corrected_{cut_name}",
            ";p_{T} [GeV/c];corrected (#Lambda+#bar{#Lambda})/(2K^{0}_{S})",
            valid_rows,
            "average_lambda_over_k0s_corrected",
            "average_lambda_over_k0s_corrected_error"
        ),
    }

    for graph in graphs.values():
        graph.Write()

    corrected_graphs_by_cut[cut_name] = graphs

corrected_output.Close()

print("Saved corrected CSV :", corrected_csv)
print("Saved corrected ROOT:", corrected_root)


Saved corrected CSV : output/v0_ratio/V0_efficiency_BR_corrected_ratios.csv
Saved corrected ROOT: output/v0_ratio/V0_efficiency_BR_corrected_ratios.root


In [28]:
# ============================================================
# STAR p+p sqrt(s) = 200 GeV
# Phys. Rev. C 75, 064901 (2007)
#
# Quantity:
#       (Lambda + anti-Lambda) / (2 K0S)
#
# Midrapidity p+p data.
# ============================================================

# STAR pT points
star_pt = array('d', [
    0.55,
    0.75,
    0.95,
    1.15,
    1.35,
    1.55,
    1.75,
    1.95,
    2.20,
    2.50,
    2.90
])

# (Lambda + anti-Lambda)/(2 K0S)
star_ratio = array('d', [
    0.18,
    0.23,
    0.29,
    0.34,
    0.39,
    0.43,
    0.46,
    0.48,
    0.49,
    0.48,
    0.45
])

# approximate total uncertainties for plotting/reference
star_ratio_err = array('d', [
    0.025,
    0.025,
    0.030,
    0.030,
    0.035,
    0.040,
    0.040,
    0.045,
    0.050,
    0.055,
    0.065
])

star_pt_err = array('d', [0.0] * len(star_pt))

g_star_pp200 = keep(root.TGraphErrors(
    len(star_pt),
    star_pt,
    star_ratio,
    star_pt_err,
    star_ratio_err
))

g_star_pp200.SetName("g_star_pp200_average_lambda_k0s")

g_star_pp200.SetMarkerStyle(33)
g_star_pp200.SetMarkerSize(1.5)
g_star_pp200.SetMarkerColor(root.kGreen + 2)
g_star_pp200.SetLineColor(root.kGreen + 2)
g_star_pp200.SetLineWidth(2)

In [29]:
# ============================================================
# Comparison plot: raw vs efficiency+BR corrected average ratio
# Existing comparison plots above are left unchanged.
# ============================================================

corrected_comparison_pdf = (
    outdir + "/V0_selected_cuts_average_lambda_k0s_efficiency_BR_corrected.pdf"
)

cuts_with_correction = [
    cut_name for cut_name in selected_cuts_to_plot
    if cut_name in corrected_graphs_by_cut
]

c_corrected = keep(root.TCanvas(
    "c_average_lambda_k0s_efficiency_BR_corrected",
    "c_average_lambda_k0s_efficiency_BR_corrected",
    1600,
    700
))
c_corrected.Divide(2, 1)

# ------------------------------------------------------------
# Pad 1: raw ratio (same physics quantity as before)
# ------------------------------------------------------------
c_corrected.cd(1)
root.gPad.SetLeftMargin(0.14)
root.gPad.SetBottomMargin(0.14)
root.gPad.SetRightMargin(0.04)
root.gPad.SetTopMargin(0.08)

raw_values = []
for cut_name in cuts_with_correction:
    for row in all_cut_results[cut_name]:
        value = row["average_lambda_over_k0s"]
        if math.isfinite(value) and math.isfinite(row["eff_k0s_over_lambda"]):
            raw_values.append(value)

frame_raw = keep(root.TH1F(
    "frame_average_lambda_k0s_raw_effcorr_comparison",
    "",
    100,
    0.5,
    3.0
))
frame_raw.SetDirectory(0)
frame_raw.SetMinimum(0.0)
frame_raw.SetMaximum(max(0.7, 1.3 * max(raw_values, default=0.5)))
Format_hist_loc(
    frame_raw,
    "",
    "p_{T} [GeV/c]",
    "raw (#Lambda+#bar{#Lambda})/(2K^{0}_{S})",
    center_title=True,
    x_title_offset=1.0,
    y_title_offset=1.10
)
frame_raw.Draw()

# ------------------------------------------------------------
# Pad 2: corrected ratio + external references already in notebook
# ------------------------------------------------------------
c_corrected.cd(2)
root.gPad.SetLeftMargin(0.14)
root.gPad.SetBottomMargin(0.14)
root.gPad.SetRightMargin(0.04)
root.gPad.SetTopMargin(0.08)

corr_values = []
for cut_name in cuts_with_correction:
    for row in all_cut_results[cut_name]:
        value = row["average_lambda_over_k0s_corrected"]
        if math.isfinite(value):
            corr_values.append(value)

external_max = 0.0
for external_graph in [g_black, g_cms]:
    for i in range(external_graph.GetN()):
        external_max = max(external_max, external_graph.GetPointY(i))

frame_corr = keep(root.TH1F(
    "frame_average_lambda_k0s_corrected_comparison",
    "",
    100,
    0.5,
    3.0
))
frame_corr.SetDirectory(0)
frame_corr.SetMinimum(0.0)
frame_corr.SetMaximum(
    max(0.8, 1.25 * max(corr_values, default=0.6), 1.25 * external_max)
)
Format_hist_loc(
    frame_corr,
    "",
    "p_{T} [GeV/c]",
    "eff.+BR corrected (#Lambda+#bar{#Lambda})/(2K^{0}_{S})",
    center_title=True,
    x_title_offset=1.0,
    y_title_offset=1.10
)
frame_corr.Draw()

# External reference points, same as the existing comparison plot.
g_cms.Draw("2 SAME")
g_cms.Draw("P SAME")
#g_black.Draw("P SAME")

legend_corr = keep(root.TLegend(0.48, 0.17, 0.92, 0.45))
legend_corr.SetBorderSize(0)
legend_corr.SetFillStyle(0)
legend_corr.SetTextFont(42)
legend_corr.SetTextSize(0.031)

for cut_index, cut_name in enumerate(cuts_with_correction):
    raw_graph = corrected_graphs_by_cut[cut_name]["average_lambda_over_k0s_raw"]
    corr_graph = corrected_graphs_by_cut[cut_name]["average_lambda_over_k0s_corrected"]

    color = comparison_colors[cut_index % len(comparison_colors)]
    marker = comparison_markers[cut_index % len(comparison_markers)]

    for graph in [raw_graph, corr_graph]:
        graph.SetMarkerColor(color)
        graph.SetLineColor(color)
        graph.SetMarkerStyle(marker)
        graph.SetMarkerSize(1.05)
        graph.SetLineWidth(2)

    c_corrected.cd(1)
    raw_graph.Draw("PZ SAME")

    c_corrected.cd(2)
    corr_graph.Draw("PZ SAME")
    legend_corr.AddEntry(corr_graph, cut_name, "lp")

c_corrected.cd(2)
legend_corr.AddEntry(g_cms, "CMS, p+p #sqrt{s}=7 TeV", "pf")
#legend_corr.AddEntry(g_black, "sPHENIX PPG16", "p")
legend_corr.Draw()

latex_corr = keep(root.TLatex())
latex_corr.SetNDC()
latex_corr.SetTextFont(42)
latex_corr.SetTextSize(0.045)
latex_corr.DrawLatex(0.16, 0.93, data_set + ", " + run_text)

latex_corr.SetTextSize(0.034)
latex_corr.DrawLatex(
    0.16,
    0.87,
    "Correction: #epsilon(K^{0}_{S})/#epsilon(#Lambda) #times BR(K^{0}_{S})/BR(#Lambda)"
)

c_corrected.Update()
c_corrected.SaveAs(corrected_comparison_pdf)
c_corrected.Draw()

print("Saved corrected comparison PDF:", corrected_comparison_pdf)
print("NOTE: cut07, 1.1--1.4 efficiency-ratio uncertainty was hidden in the supplied screenshot;")
print("      its corrected central value is calculated, but that point's error does not include efficiency uncertainty.")


Saved corrected comparison PDF: output/v0_ratio/V0_selected_cuts_average_lambda_k0s_efficiency_BR_corrected.pdf
NOTE: cut07, 1.1--1.4 efficiency-ratio uncertainty was hidden in the supplied screenshot;
      its corrected central value is calculated, but that point's error does not include efficiency uncertainty.


In [30]:
# ============================================================
# Fully corrected K0S and Lambda invariant spectra
# + reference Tsallis functions
# ============================================================

from array import array
import math
import ROOT as root

# ------------------------------------------------------------
# DATA NORMALIZATION
# ------------------------------------------------------------

print("N_EVENTS =", N_EVENTS)

# IMPORTANT:
# Set this to your actual reconstructed rapidity interval.
#
# Examples:
#   |y| < 0.5  -> DELTA_Y = 1.0
#   |y| < 1.0  -> DELTA_Y = 2.0
#
DELTA_Y = 1.0

# Effective cross section represented by the event sample.
#
# Needed because the PHENIX reference Tsallis curves are cross sections
# in mb.
#
# Replace this with the proper value for your data sample if necessary.
SIGMA_EVENT_MB = 42.0


# ------------------------------------------------------------
# BRANCHING RATIOS
# ------------------------------------------------------------

BR_K0S = 0.6920                 # K0S -> pi+ pi-
BR_LAMBDA = 0.6390              # Lambda -> p pi-





# ------------------------------------------------------------
# Find efficiency corresponding to pT
# returns FRACTION, not percent
# ------------------------------------------------------------

def get_efficiency(eff_table, cut_name, ptmin, ptmax=None):
    """Return efficiency as a fraction. Prefer a full [ptmin,ptmax] interval."""
    if ptmax is not None:
        return effective_efficiency(eff_table, cut_name, ptmin, ptmax)

    # Backward-compatible point lookup for any legacy call.
    values = eff_table.get(cut_name)
    if values is None:
        return None
    for i, ((lo, hi), entry) in enumerate(zip(eff_pt_bins, values)):
        inside = lo <= ptmin < hi or (i == len(eff_pt_bins)-1 and lo <= ptmin <= hi)
        if inside:
            if entry is None:
                return None
            return entry[0]/100.0, entry[1]/100.0
    return None

def corrected_invariant_spectrum(
    raw_yield,
    raw_error,
    pt,
    dpt,
    efficiency,
    efficiency_error,
    branching_ratio
):

    if (
        raw_yield <= 0 or
        efficiency <= 0 or
        pt <= 0 or
        dpt <= 0
    ):
        return float("nan"), float("nan")

    denominator = (
        N_EVENTS
        * efficiency
        * branching_ratio
        * dpt
        * DELTA_Y
        * 2.0
        * math.pi
        * pt
    )

    value = (
        raw_yield
        / denominator
        * SIGMA_EVENT_MB
    )

    # raw statistical + MC efficiency statistical uncertainty
    rel2 = 0.0

    if raw_error > 0:
        rel2 += (raw_error / raw_yield)**2

    if efficiency_error > 0:
        rel2 += (efficiency_error / efficiency)**2

    error = abs(value) * math.sqrt(rel2)

    return value, error


# ============================================================
# Tsallis
#
# same convention as your K0S function:
#
#   1/(2 pi pT) d2sigma/(dpT dy)
# ============================================================

def TsallisFit(x, par):

    pt = x[0]

    m0 = par[0]
    dsdy = par[1]
    T = par[2]
    n = par[3]

    mt = math.sqrt(pt*pt + m0*m0)

    norm = (
        (n - 1.0) * (n - 2.0)
        /
        (
            (n*T + m0*(n - 1.0))
            *
            (n*T + m0)
        )
    )

    return (
        1.0 / (2.0 * math.pi)
        * dsdy
        * norm
        * ((n*T + mt) / (n*T + m0))**(-n)
    )


# ------------------------------------------------------------
# K0S Tsallis from your fit
# ------------------------------------------------------------

f_k0s_tsallis = keep(root.TF1(
    "f_k0s_tsallis_reference",
    TsallisFit,
    0.5,
    3.0,
    4
))

f_k0s_tsallis.SetParameters(
    0.493680,     # mass
    3.73504,      # dsigma/dy [mb]
    0.114870,     # T [GeV]
    8.34434       # n
)


# ------------------------------------------------------------
# Proton PHENIX Tsallis
#
# Table VIII:
# dsigma/dy = 1.76 mb
# T         = 58.8 MeV
# n         = 9.20
# ------------------------------------------------------------

f_proton_tsallis = keep(root.TF1(
    "f_proton_tsallis_reference",
    TsallisFit,
    0.5,
    3.0,
    4
))

f_proton_tsallis.SetParameters(
    0.938272,     # proton mass
    1.76,         # dsigma/dy [mb]
    0.0588,       # T = 58.8 MeV = 0.0588 GeV
    9.20
)


# ============================================================
# mT-scaled proton -> Lambda reference
#
# At Lambda pT:
#
# mT_Lambda = sqrt(pT_Lambda^2 + mLambda^2)
#
# find proton pT with the SAME mT:
#
# pT_proton = sqrt(mT_Lambda^2 - mp^2)
#
# Evaluate proton Tsallis there.
#
# An overall normalization is left configurable.
# ============================================================

M_PROTON = 0.938272
M_LAMBDA = 1.115683

LAMBDA_MT_SCALE = 1.0


def proton_mt_scaled_to_lambda(x, par):

    pt_lambda = x[0]

    mt_lambda = math.sqrt(
        pt_lambda*pt_lambda +
        M_LAMBDA*M_LAMBDA
    )

    pt_proton_sq = (
        mt_lambda*mt_lambda -
        M_PROTON*M_PROTON
    )

    if pt_proton_sq <= 0:
        return 0.0

    pt_proton = math.sqrt(pt_proton_sq)

    return (
        LAMBDA_MT_SCALE
        * f_proton_tsallis.Eval(pt_proton)
    )


f_lambda_mt_scaled = keep(root.TF1(
    "f_lambda_mt_scaled_from_proton",
    proton_mt_scaled_to_lambda,
    0.5,
    3.0,
    0
))

print("Correction setup ready.")
print("N events =", N_EVENTS)
print("Delta y =", DELTA_Y)
print("sigma_event =", SIGMA_EVENT_MB, "mb")

N_EVENTS = 380000000
Correction setup ready.
N events = 380000000
Delta y = 1.0
sigma_event = 42.0 mb


In [31]:
# ============================================================
# Construct corrected spectra for every cut
# ============================================================

corrected_spectra = {}

for cut_name in selected_cuts_to_plot:

    if cut_name not in all_cut_results:
        continue

    rows = all_cut_results[cut_name]

    x_k = []
    ex_k = []
    y_k = []
    ey_k = []

    x_l = []
    ex_l = []
    y_l = []
    ey_l = []

    for row in rows:

        pt = row["pt_center"]
        dpt = 2.0 * row["pt_half_width"]

        # ================================================
        # K0S
        # ================================================

        k_eff_info = get_efficiency(
            k0s_eff_percent,
            cut_name,
            row["ptmin"],
            row["ptmax"]
        )

        if k_eff_info is not None:

            eff, eff_err = k_eff_info

            value, error = corrected_invariant_spectrum(
                row["k0s_yield"],
                row["k0s_yield_error"],
                pt,
                dpt,
                eff,
                eff_err,
                BR_K0S
            )

            if math.isfinite(value):

                x_k.append(pt)
                ex_k.append(row["pt_half_width"])
                y_k.append(value)
                ey_k.append(error)

                # save into original row as well
                row["k0s_efficiency"] = eff
                row["k0s_efficiency_error"] = eff_err

                row["k0s_corrected_spectrum"] = value
                row["k0s_corrected_spectrum_error"] = error

        # ================================================
        # Lambda
        # ================================================

        l_eff_info = get_efficiency(
            lambda_eff_percent,
            cut_name,
            row["ptmin"],
            row["ptmax"]
        )

        if l_eff_info is not None:

            eff, eff_err = l_eff_info

            value, error = corrected_invariant_spectrum(
                row["lambda_yield"],
                row["lambda_yield_error"],
                pt,
                dpt,
                eff,
                eff_err,
                BR_LAMBDA
            )

            if math.isfinite(value):

                x_l.append(pt)
                ex_l.append(row["pt_half_width"])
                y_l.append(value)
                ey_l.append(error)

                row["lambda_efficiency"] = eff
                row["lambda_efficiency_error"] = eff_err

                row["lambda_corrected_spectrum"] = value
                row["lambda_corrected_spectrum_error"] = error


    # ----------------------------------------------------
    # ROOT graphs
    # ----------------------------------------------------

    g_k = keep(root.TGraphErrors(
        len(x_k),
        array('d', x_k),
        array('d', y_k),
        array('d', ex_k),
        array('d', ey_k)
    ))

    g_k.SetName(
        f"g_k0s_fully_corrected_{cut_name}"
    )


    g_l = keep(root.TGraphErrors(
        len(x_l),
        array('d', x_l),
        array('d', y_l),
        array('d', ex_l),
        array('d', ey_l)
    ))

    g_l.SetName(
        f"g_lambda_fully_corrected_{cut_name}"
    )


    corrected_spectra[cut_name] = {
        "k0s": g_k,
        "lambda": g_l,
    }


print("Built fully corrected spectra for:")
for cut_name in corrected_spectra:
    print("  ", cut_name)

Built fully corrected spectra for:
   cut02_preselection
   cut03_baseline
   cut04_pairDCA_15mm
   cut05_pairDCA_10mm
   cut06_pairDCA_7mm
   cut07_pairDCA_5mm


In [32]:
# ============================================================
# Plot corrected spectra + reference Tsallis
#
# TOP:
#   K0S corrected data
#   Lambda corrected data
#   K0S Tsallis
#   proton -> Lambda mT-scaled curve
#
# BOTTOM:
#   K0S / K0S Tsallis
#   Lambda / mT-scaled proton
# ============================================================

spectrum_canvases = {}

for cut_index, cut_name in enumerate(selected_cuts_to_plot):

    if cut_name not in corrected_spectra:
        continue

    g_k = corrected_spectra[cut_name]["k0s"]
    g_l = corrected_spectra[cut_name]["lambda"]

    # --------------------------------------------------------
    # Canvas
    # --------------------------------------------------------

    c = keep(root.TCanvas(
        f"c_corrected_spectrum_{cut_name}",
        cut_name,
        900,
        900
    ))

    # upper spectrum pad
    pad1 = keep(root.TPad(
        f"pad1_{cut_name}",
        "",
        0.0,
        0.30,
        1.0,
        1.0
    ))

    pad1.SetBottomMargin(0.02)
    pad1.SetLeftMargin(0.15)
    pad1.SetRightMargin(0.04)
    pad1.SetTopMargin(0.07)
    pad1.SetLogy()

    pad1.Draw()


    # lower ratio pad
    pad2 = keep(root.TPad(
        f"pad2_{cut_name}",
        "",
        0.0,
        0.0,
        1.0,
        0.30
    ))

    pad2.SetTopMargin(0.02)
    pad2.SetBottomMargin(0.32)
    pad2.SetLeftMargin(0.15)
    pad2.SetRightMargin(0.04)

    pad2.Draw()


    # ========================================================
    # TOP
    # ========================================================

    pad1.cd()

    frame = keep(root.TH1F(
        f"frame_corrected_spectrum_{cut_name}",
        "",
        100,
        0.7,
        3.05
    ))

    frame.SetDirectory(0)

    frame.SetMinimum(1.0e-5)
    frame.SetMaximum(10.0)

    Format_hist_loc(
        frame,
        "",
        "",
        "#frac{1}{2#pi p_{T}} #frac{d^{2}#sigma}{dp_{T}dy}  [mb/(GeV/c)^{2}]",
        center_title=True,
        x_title_offset=1.0,
        y_title_offset=1.25
    )

    frame.GetXaxis().SetLabelSize(0)
    frame.Draw()


    # --------------------------------------------------------
    # Data styles
    # --------------------------------------------------------

    g_k.SetMarkerStyle(20)
    g_k.SetMarkerSize(1.25)
    g_k.SetMarkerColor(root.kBlue + 1)
    g_k.SetLineColor(root.kBlue + 1)
    g_k.SetLineWidth(2)

    g_l.SetMarkerStyle(21)
    g_l.SetMarkerSize(1.25)
    g_l.SetMarkerColor(root.kRed + 1)
    g_l.SetLineColor(root.kRed + 1)
    g_l.SetLineWidth(2)


    # reference curves
    f_k0s_tsallis.SetLineColor(root.kBlue + 1)
    f_k0s_tsallis.SetLineWidth(3)
    f_k0s_tsallis.SetLineStyle(2)

    f_lambda_mt_scaled.SetLineColor(root.kRed + 1)
    f_lambda_mt_scaled.SetLineWidth(3)
    f_lambda_mt_scaled.SetLineStyle(2)


    f_k0s_tsallis.Draw("SAME")
    f_lambda_mt_scaled.Draw("SAME")

    g_k.Draw("PZ SAME")
    g_l.Draw("PZ SAME")


    # --------------------------------------------------------
    # Legend
    # --------------------------------------------------------

    leg = keep(root.TLegend(
        0.49,
        0.61,
        0.91,
        0.90
    ))

    leg.SetBorderSize(0)
    leg.SetFillStyle(0)
    leg.SetTextFont(42)
    leg.SetTextSize(0.033)

    leg.AddEntry(
        g_k,
        "K^{0}_{S} corrected",
        "lp"
    )

    leg.AddEntry(
        g_l,
        "#Lambda corrected",
        "lp"
    )

    leg.AddEntry(
        f_k0s_tsallis,
        "K^{0}_{S} Tsallis",
        "l"
    )

    leg.AddEntry(
        f_lambda_mt_scaled,
        "p Tsallis, m_{T}-scaled to #Lambda",
        "l"
    )

    leg.Draw()


    tex = keep(root.TLatex())
    tex.SetNDC()
    tex.SetTextFont(42)
    tex.SetTextSize(0.043)

    tex.DrawLatex(
        0.17,
        0.93,
        data_set + ", " + run_text
    )

    tex.SetTextSize(0.032)

    tex.DrawLatex(
        0.17,
        0.875,
        cut_name
    )


    # ========================================================
    # BOTTOM: DATA / REFERENCE
    # ========================================================

    pad2.cd()

    ratio_frame = keep(root.TH1F(
        f"frame_corrected_ratio_{cut_name}",
        "",
        100,
        0.7,
        3.05
    ))

    ratio_frame.SetDirectory(0)

    ratio_frame.SetMinimum(0.0)
    ratio_frame.SetMaximum(2.5)

    Format_hist_loc(
        ratio_frame,
        "",
        "p_{T} [GeV/c]",
        "Data / reference",
        center_title=True,
        x_title_offset=1.0,
        y_title_offset=0.60
    )

    ratio_frame.GetXaxis().SetTitleSize(0.11)
    ratio_frame.GetXaxis().SetLabelSize(0.09)

    ratio_frame.GetYaxis().SetTitleSize(0.095)
    ratio_frame.GetYaxis().SetLabelSize(0.08)
    ratio_frame.GetYaxis().SetNdivisions(505)

    ratio_frame.Draw()


    # --------------------------------------------------------
    # helper: divide graph by TF1
    # --------------------------------------------------------

    def graph_divide_function(graph, func, name):

        n = graph.GetN()

        xx = []
        yy = []
        exx = []
        eyy = []

        for i in range(n):

            x = graph.GetPointX(i)
            y = graph.GetPointY(i)

            ex = graph.GetErrorX(i)
            ey = graph.GetErrorY(i)

            ref = func.Eval(x)

            if ref <= 0:
                continue

            xx.append(x)
            exx.append(ex)

            yy.append(y / ref)
            eyy.append(ey / ref)

        g = keep(root.TGraphErrors(
            len(xx),
            array('d', xx),
            array('d', yy),
            array('d', exx),
            array('d', eyy)
        ))

        g.SetName(name)

        return g


    g_k_ratio = graph_divide_function(
        g_k,
        f_k0s_tsallis,
        f"g_k0s_over_tsallis_{cut_name}"
    )

    g_l_ratio = graph_divide_function(
        g_l,
        f_lambda_mt_scaled,
        f"g_lambda_over_mt_scaled_proton_{cut_name}"
    )


    g_k_ratio.SetMarkerStyle(20)
    g_k_ratio.SetMarkerSize(1.15)
    g_k_ratio.SetMarkerColor(root.kBlue + 1)
    g_k_ratio.SetLineColor(root.kBlue + 1)

    g_l_ratio.SetMarkerStyle(21)
    g_l_ratio.SetMarkerSize(1.15)
    g_l_ratio.SetMarkerColor(root.kRed + 1)
    g_l_ratio.SetLineColor(root.kRed + 1)


    g_k_ratio.Draw("PZ SAME")
    g_l_ratio.Draw("PZ SAME")


    line1 = keep(root.TLine(
        0.7,
        1.0,
        3.05,
        1.0
    ))

    line1.SetLineStyle(2)
    line1.SetLineWidth(2)
    line1.Draw()


    c.cd()
    c.Update()

    spectrum_canvases[cut_name] = c

    #c.Draw()

In [33]:
# ============================================================
# Summary comparison:
#
# Canvas 1:
#   K0S corrected spectra for ALL selected cuts
#   + K0S Tsallis
#   + Data / Tsallis ratio panel
#
# Canvas 2:
#   Lambda corrected spectra for ALL selected cuts
#   + proton mT-scaled reference
#   + Data / reference ratio panel
# ============================================================

from array import array
import math
import ROOT as root


cuts_to_draw = [
    #"cut02_preselection",
    "cut03_baseline",
    "cut04_pairDCA_15mm",
    "cut05_pairDCA_10mm",
    "cut06_pairDCA_7mm",
    "cut07_pairDCA_5mm",
]

# ------------------------------------------------------------
# helper: graph / TF1
# ------------------------------------------------------------

def graph_divide_function_summary(graph, func, name):

    xx = []
    yy = []
    exx = []
    eyy = []

    for i in range(graph.GetN()):

        x = graph.GetPointX(i)
        y = graph.GetPointY(i)

        ex = graph.GetErrorX(i)
        ey = graph.GetErrorY(i)

        ref = func.Eval(x)

        if not math.isfinite(ref) or ref <= 0:
            continue

        xx.append(x)
        exx.append(ex)

        yy.append(y / ref)
        eyy.append(ey / ref)

    g = keep(root.TGraphErrors(
        len(xx),
        array('d', xx),
        array('d', yy),
        array('d', exx),
        array('d', eyy)
    ))

    g.SetName(name)

    return g


# ------------------------------------------------------------
# only cuts which actually have corrected spectra
# ------------------------------------------------------------

valid_cuts_spectra = [
    cut_name
    for cut_name in cuts_to_draw
    if cut_name in corrected_spectra
]

print("Cuts drawn:")
for cut_name in valid_cuts_spectra:
    print("  ", cut_name)


# ============================================================
# 1. K0S CANVAS
# ============================================================

c_k0s_allcuts = keep(root.TCanvas(
    "c_k0s_corrected_allcuts_vs_tsallis",
    "K0S corrected spectra all cuts",
    950,
    900
))


# ------------------------------------------------------------
# pads
# ------------------------------------------------------------

pad_k_top = keep(root.TPad(
    "pad_k_top",
    "",
    0.0,
    0.30,
    1.0,
    1.0
))

pad_k_top.SetLeftMargin(0.15)
pad_k_top.SetRightMargin(0.04)
pad_k_top.SetTopMargin(0.07)
pad_k_top.SetBottomMargin(0.02)
pad_k_top.SetLogy()
pad_k_top.Draw()


pad_k_bottom = keep(root.TPad(
    "pad_k_bottom",
    "",
    0.0,
    0.0,
    1.0,
    0.30
))

pad_k_bottom.SetLeftMargin(0.15)
pad_k_bottom.SetRightMargin(0.04)
pad_k_bottom.SetTopMargin(0.02)
pad_k_bottom.SetBottomMargin(0.32)
pad_k_bottom.Draw()


# ============================================================
# K0S TOP PANEL
# ============================================================

pad_k_top.cd()

frame_k = keep(root.TH1F(
    "frame_k0s_allcuts",
    "",
    100,
    0.75,
    3.05
))

frame_k.SetDirectory(0)
frame_k.SetMinimum(1.0e-5)
frame_k.SetMaximum(10.0)

Format_hist_loc(
    frame_k,
    "",
    "",
    "#frac{1}{2#pi p_{T}} #frac{d^{2}#sigma}{dp_{T}dy}  [mb/(GeV/c)^{2}]",
    center_title=True,
    x_title_offset=1.0,
    y_title_offset=1.25
)

frame_k.GetXaxis().SetLabelSize(0)
frame_k.Draw()


# reference curve
f_k0s_tsallis.SetLineColor(root.kBlack)
f_k0s_tsallis.SetLineWidth(4)
f_k0s_tsallis.SetLineStyle(1)

f_k0s_tsallis.Draw("SAME")


# legend
leg_k = keep(root.TLegend(
    0.56,
    0.51,
    0.92,
    0.90
))

leg_k.SetBorderSize(0)
leg_k.SetFillStyle(0)
leg_k.SetTextFont(42)
leg_k.SetTextSize(0.030)

leg_k.AddEntry(
    f_k0s_tsallis,
    "K^{0}_{S} Tsallis reference",
    "l"
)


k0s_ratio_graphs = {}


# ------------------------------------------------------------
# draw all cuts
# ------------------------------------------------------------

for cut_index, cut_name in enumerate(valid_cuts_spectra):

    graph = corrected_spectra[cut_name]["k0s"]

    color = comparison_colors[
        cut_index % len(comparison_colors)
    ]

    marker = comparison_markers[
        cut_index % len(comparison_markers)
    ]

    graph.SetMarkerColor(color)
    graph.SetLineColor(color)
    graph.SetMarkerStyle(marker)
    graph.SetMarkerSize(1.15)
    graph.SetLineWidth(2)

    graph.Draw("PZ SAME")

    leg_k.AddEntry(
        graph,
        cut_name,
        "lp"
    )

    # ratio graph
    g_ratio = graph_divide_function_summary(
        graph,
        f_k0s_tsallis,
        f"g_k0s_ratio_summary_{cut_name}"
    )

    g_ratio.SetMarkerColor(color)
    g_ratio.SetLineColor(color)
    g_ratio.SetMarkerStyle(marker)
    g_ratio.SetMarkerSize(1.05)
    g_ratio.SetLineWidth(2)

    k0s_ratio_graphs[cut_name] = g_ratio


leg_k.Draw()


# labels
latex_k = keep(root.TLatex())
latex_k.SetNDC()
latex_k.SetTextFont(42)

latex_k.SetTextSize(0.043)
latex_k.DrawLatex(
    0.17,
    0.93,
    data_set + ", " + run_text
)

latex_k.SetTextSize(0.035)
latex_k.DrawLatex(
    0.17,
    0.875,
    "K^{0}_{S}"
)


# ============================================================
# K0S RATIO PANEL
# ============================================================

pad_k_bottom.cd()

frame_k_ratio = keep(root.TH1F(
    "frame_k0s_ratio_allcuts",
    "",
    100,
    0.75,
    3.05
))

frame_k_ratio.SetDirectory(0)
frame_k_ratio.SetMinimum(0.0)
frame_k_ratio.SetMaximum(2.5)

Format_hist_loc(
    frame_k_ratio,
    "",
    "p_{T} [GeV/c]",
    "Data / Tsallis",
    center_title=True,
    x_title_offset=1.0,
    y_title_offset=0.58
)

frame_k_ratio.GetXaxis().SetTitleSize(0.11)
frame_k_ratio.GetXaxis().SetLabelSize(0.09)

frame_k_ratio.GetYaxis().SetTitleSize(0.095)
frame_k_ratio.GetYaxis().SetLabelSize(0.08)
frame_k_ratio.GetYaxis().SetNdivisions(505)

frame_k_ratio.Draw()


line_k_one = keep(root.TLine(
    0.75,
    1.0,
    3.05,
    1.0
))

line_k_one.SetLineStyle(2)
line_k_one.SetLineWidth(2)
line_k_one.Draw()


for cut_name in valid_cuts_spectra:
    k0s_ratio_graphs[cut_name].Draw("PZ SAME")


c_k0s_allcuts.cd()
c_k0s_allcuts.Update()
c_k0s_allcuts.Draw()


# ============================================================
# 2. LAMBDA CANVAS
# ============================================================

c_lambda_allcuts = keep(root.TCanvas(
    "c_lambda_corrected_allcuts_vs_mt_scaled_proton",
    "Lambda corrected spectra all cuts",
    950,
    900
))


# ------------------------------------------------------------
# pads
# ------------------------------------------------------------

pad_l_top = keep(root.TPad(
    "pad_l_top",
    "",
    0.0,
    0.30,
    1.0,
    1.0
))

pad_l_top.SetLeftMargin(0.15)
pad_l_top.SetRightMargin(0.04)
pad_l_top.SetTopMargin(0.07)
pad_l_top.SetBottomMargin(0.02)
pad_l_top.SetLogy()
pad_l_top.Draw()


pad_l_bottom = keep(root.TPad(
    "pad_l_bottom",
    "",
    0.0,
    0.0,
    1.0,
    0.30
))

pad_l_bottom.SetLeftMargin(0.15)
pad_l_bottom.SetRightMargin(0.04)
pad_l_bottom.SetTopMargin(0.02)
pad_l_bottom.SetBottomMargin(0.32)
pad_l_bottom.Draw()


# ============================================================
# LAMBDA TOP PANEL
# ============================================================

pad_l_top.cd()

frame_l = keep(root.TH1F(
    "frame_lambda_allcuts",
    "",
    100,
    0.75,
    3.05
))

frame_l.SetDirectory(0)
frame_l.SetMinimum(1.0e-5)
frame_l.SetMaximum(10.0)

Format_hist_loc(
    frame_l,
    "",
    "",
    "#frac{1}{2#pi p_{T}} #frac{d^{2}#sigma}{dp_{T}dy}  [mb/(GeV/c)^{2}]",
    center_title=True,
    x_title_offset=1.0,
    y_title_offset=1.25
)

frame_l.GetXaxis().SetLabelSize(0)
frame_l.Draw()


# reference
f_lambda_mt_scaled.SetLineColor(root.kBlack)
f_lambda_mt_scaled.SetLineWidth(4)
f_lambda_mt_scaled.SetLineStyle(1)

f_lambda_mt_scaled.Draw("SAME")


# legend
leg_l = keep(root.TLegend(
    0.53,
    0.51,
    0.92,
    0.90
))

leg_l.SetBorderSize(0)
leg_l.SetFillStyle(0)
leg_l.SetTextFont(42)
leg_l.SetTextSize(0.030)

leg_l.AddEntry(
    f_lambda_mt_scaled,
    "p Tsallis, m_{T}-scaled to #Lambda",
    "l"
)


lambda_ratio_graphs = {}


# ------------------------------------------------------------
# draw all cuts
# ------------------------------------------------------------

for cut_index, cut_name in enumerate(valid_cuts_spectra):

    graph = corrected_spectra[cut_name]["lambda"]

    color = comparison_colors[
        cut_index % len(comparison_colors)
    ]

    marker = comparison_markers[
        cut_index % len(comparison_markers)
    ]

    graph.SetMarkerColor(color)
    graph.SetLineColor(color)
    graph.SetMarkerStyle(marker)
    graph.SetMarkerSize(1.15)
    graph.SetLineWidth(2)

    graph.Draw("PZ SAME")

    leg_l.AddEntry(
        graph,
        cut_name,
        "lp"
    )

    # ratio
    g_ratio = graph_divide_function_summary(
        graph,
        f_lambda_mt_scaled,
        f"g_lambda_ratio_summary_{cut_name}"
    )

    g_ratio.SetMarkerColor(color)
    g_ratio.SetLineColor(color)
    g_ratio.SetMarkerStyle(marker)
    g_ratio.SetMarkerSize(1.05)
    g_ratio.SetLineWidth(2)

    lambda_ratio_graphs[cut_name] = g_ratio


leg_l.Draw()


latex_l = keep(root.TLatex())
latex_l.SetNDC()
latex_l.SetTextFont(42)

latex_l.SetTextSize(0.043)
latex_l.DrawLatex(
    0.17,
    0.93,
    data_set + ", " + run_text
)

latex_l.SetTextSize(0.035)
latex_l.DrawLatex(
    0.17,
    0.875,
    "#Lambda"
)


# ============================================================
# LAMBDA RATIO PANEL
# ============================================================

pad_l_bottom.cd()

frame_l_ratio = keep(root.TH1F(
    "frame_lambda_ratio_allcuts",
    "",
    100,
    0.75,
    3.05
))

frame_l_ratio.SetDirectory(0)
frame_l_ratio.SetMinimum(0.0)
frame_l_ratio.SetMaximum(2.5)

Format_hist_loc(
    frame_l_ratio,
    "",
    "p_{T} [GeV/c]",
    "Data / reference",
    center_title=True,
    x_title_offset=1.0,
    y_title_offset=0.58
)

frame_l_ratio.GetXaxis().SetTitleSize(0.11)
frame_l_ratio.GetXaxis().SetLabelSize(0.09)

frame_l_ratio.GetYaxis().SetTitleSize(0.095)
frame_l_ratio.GetYaxis().SetLabelSize(0.08)
frame_l_ratio.GetYaxis().SetNdivisions(505)

frame_l_ratio.Draw()


line_l_one = keep(root.TLine(
    0.75,
    1.0,
    3.05,
    1.0
))

line_l_one.SetLineStyle(2)
line_l_one.SetLineWidth(2)
line_l_one.Draw()


for cut_name in valid_cuts_spectra:
    lambda_ratio_graphs[cut_name].Draw("PZ SAME")


c_lambda_allcuts.cd()
c_lambda_allcuts.Update()
c_lambda_allcuts.Draw()


# ============================================================
# save
# ============================================================

k0s_summary_pdf = (
    outdir
    + "/K0S_corrected_spectra_all_selected_cuts_vs_Tsallis.pdf"
)

lambda_summary_pdf = (
    outdir
    + "/Lambda_corrected_spectra_all_selected_cuts_vs_mt_scaled_proton.pdf"
)

c_k0s_allcuts.SaveAs(k0s_summary_pdf)
c_lambda_allcuts.SaveAs(lambda_summary_pdf)

print("Saved:")
print(" ", k0s_summary_pdf)
print(" ", lambda_summary_pdf)

Cuts drawn:
   cut03_baseline
   cut04_pairDCA_15mm
   cut05_pairDCA_10mm
   cut06_pairDCA_7mm
   cut07_pairDCA_5mm
Saved:
  output/v0_ratio/K0S_corrected_spectra_all_selected_cuts_vs_Tsallis.pdf
  output/v0_ratio/Lambda_corrected_spectra_all_selected_cuts_vs_mt_scaled_proton.pdf
